In [9]:
import os, sys, subprocess, platform

# Tampilkan informasi CPU & RAM (menyesuaikan OS Linux Kaggle/Colab atau Windows lokal)
if sys.platform != "win32":
    !lscpu | grep "Model name\|CPU(s):"
    !free -h
else:
    print(f"OS: {platform.system()} {platform.release()} | CPU: {platform.processor()}")

# Hanya install open_clip_torch jika belum terpasang
try:
    import open_clip
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "open_clip_torch"])


CPU(s):                                  4
Model name:                              Intel(R) Xeon(R) CPU @ 2.20GHz
NUMA node0 CPU(s):                       0-3
               total        used        free      shared  buff/cache   available
Mem:            31Gi       1.7Gi        24Gi       1.2Mi       5.7Gi        29Gi
Swap:             0B          0B          0B


In [10]:
# add dependencies ini untuk model varian , uncomment untuk menjalankan
# !pip install -q --upgrade open_clip_torch transformers sentence-transformers sentencepiece protobuf

In [11]:
import time
start_time = time.time()

In [12]:
import pandas as pd, torch
import matplotlib.pyplot as plt, textwrap
import numpy as np
from PIL import Image
from IPython.display import display, Image as IPyImage

# Uncomment if use online drive (collab)
#from google.colab import drive

# modul tambahan, untuk mengacak data secara konsisten dan  membersihkan memori RAM secara manual
import gc, random
import torch.nn.functional as F

# Library encoder untuk eksperimen LAION-only.
import open_clip
# [NONAKTIF - tidak dipakai pada revisi LAION-only]
# from sentence_transformers import SentenceTransformer
# from transformers import AutoModel, AutoProcessor

# modul untuk membuka & membaca file zip, json
import zipfile
import json

# untuk mengubah data mentah gambar (bytes) yang dibaca langsung dari dalam .zip menjadi format yang bisa dibuka oleh Python/PIL
from io import BytesIO

# SEED kunci angka acak(identik untuk semua eksperimen)
SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"Environment siap. PyTorch version: {torch.__version__} | Device: CPU | SEED: {SEED}")

Environment siap. PyTorch version: 2.11.0+cpu | Device: CPU | SEED: 0


In [13]:
# Uncomment these if use online drive (collab)
# Note: Sesuaiin path_train, path_val, path_images di bawah ini sesuai dengan lokasi file di drive kalian

#from google.colab import drive
#drive.mount("/content/drive")
#BASE_DIR    = "/content/drive/MyDrive/Proyek-Akhir-PMM-Sigma"
#PATH_TRAIN  = f"{BASE_DIR}/data/"
#PATH_VAL    = f"{BASE_DIR}/data/"
#PATH_IMAGES = f"{BASE_DIR}/data/"
#PATH_OUTPUT = f"{BASE_DIR}/Output/hasil_evaluasi.json"
#PATH_CACHE  = f"{BASE_DIR}/Output/cache_m2/"
#EXPORT_DIR  = f"{BASE_DIR}/Output/M2_Experiment_Results"
#os.makedirs(os.path.dirname(PATH_OUTPUT), exist_ok=True)
#os.makedirs(PATH_CACHE, exist_ok=True)
#os.makedirs(EXPORT_DIR, exist_ok=True)


In [14]:
# Uncomment these if use kaggle
# Note: Sesuaiin path_train, path_val, path_images di bawah ini sesuai dengan lokasi file di datasets kalian
PATH_TRAIN = "/kaggle/input/datasets/willysyifaluthfia/dataset-multimodal-sigma/DATASET MULTIMODAL SIGMA/"
PATH_VAL = "/kaggle/input/datasets/willysyifaluthfia/dataset-multimodal-sigma/DATASET MULTIMODAL SIGMA/"
PATH_IMAGES = "/kaggle/input/datasets/willysyifaluthfia/dataset-multimodal-sigma/DATASET MULTIMODAL SIGMA/"

PATH_OUTPUT = "/kaggle/working/hasil_evaluasi.json"

# Fungsi untuk menyimpan hasil vektor embbedding, agar tidak menghitung ulang
PATH_CACHE  = "/kaggle/working/cache_m2/"
EXPORT_DIR  = "/kaggle/working/M2_Experiment_Results"
os.makedirs(PATH_CACHE, exist_ok=True)
os.makedirs(EXPORT_DIR, exist_ok=True)


In [15]:
# Uncomment these if use local
#BASE_DATA   = "../data/" if os.path.exists("../data/val_clean.csv") else "data/"
#BASE_OUT    = "../Output/" if os.path.exists("../data/val_clean.csv") else "Output/"
#PATH_TRAIN  = f"{BASE_DATA}"
#PATH_VAL    = f"{BASE_DATA}"
#PATH_IMAGES = f"{BASE_DATA}"
#PATH_OUTPUT = f"{BASE_OUT}hasil_evaluasi.json"
#PATH_CACHE  = f"{BASE_OUT}cache_m2/"
#EXPORT_DIR  = f"{BASE_OUT}M2_Experiment_Results"
#os.makedirs(os.path.dirname(PATH_OUTPUT), exist_ok=True)
#os.makedirs(PATH_CACHE, exist_ok=True)
#os.makedirs(EXPORT_DIR, exist_ok=True)


In [16]:
# Read train_clean.csv
# df = pd.read_csv("train_clean.csv")

# Read val_clean.csv
df = pd.read_csv(PATH_VAL + "val_clean.csv")
df_new = df #print(df_new["file"].head)
df_new["file"] = [(PATH_IMAGES + f) for f in df_new["file"]] #print(df_new["file"].head)
imgs  = [Image.open(f).convert("RGB") for f in df_new["file"]]
cap_id = list(df["caption_id"])
cap_en = list(df["caption_en"])

# Keluaran untuk memastikan dataset
print(f"Berhasil memuat {len(imgs)} gambar, {len(cap_id)} caption ID, dan {len(cap_en)} caption EN.")

Berhasil memuat 275 gambar, 275 caption ID, dan 275 caption EN.


# Inisialisasi Model — Revisi LAION-only

Notebook ini menjalankan satu model saja: **LAION-5B XLM-RoBERTa-Base ViT-B-32** melalui OpenCLIP. Kode eksperimen multi-model sebelumnya tetap disimpan sebagai komentar untuk menjaga jejak perubahan.


## Log perubahan — 25 November 2026

- Sel multi-model lama dipertahankan sebagai komentar dengan label **ARSIP NONAKTIF**.
- Sel pengganti ditambahkan tepat setelah sel arsip agar urutan eksekusi tetap benar.
- Hanya checkpoint LAION `laion5b_s13b_b90k` yang dijalankan pada M2 dan M3.
- Output eksperimen dipisahkan ke `/kaggle/working/LAION_Only_Experiment_Results/` agar tidak mencampur hasil dengan eksperimen sebelumnya.
- Konfigurasi CPU dipertahankan seperti kode awal.


In [17]:
# [ARSIP NONAKTIF: Implementasi encoder dan manager empat model sebelumnya; diganti oleh sel LAION-only berikutnya.]

# # Ekstraksi Model Google SigLIP 2
# def encode_siglip2_hf(model_id, imgs, cap_id, cap_en, img_bs=32, txt_bs=64): # Pemrosesan Per Batch
#     """Ekstraksi untuk Varian 2: google/siglip2-base-patch16-224 via transformers"""
#     processor = AutoProcessor.from_pretrained(model_id)
#     model = AutoModel.from_pretrained(model_id)
#     model.eval()
#     model.requires_grad_(False) # Untuk Memastikan seluruh bobot pretrained tetap frozen
#         
#
#     img_embs, ti_embs, te_embs = [], [], []
#
#     with torch.no_grad():
#         # Ekstraksi embedding gambar
#         for i in range(0, len(imgs), img_bs):
#             inputs = processor(
#                 images=imgs[i : i + img_bs],
#                 return_tensors="pt"
#             )
#
#             outputs = model.get_image_features(**inputs)
#             feats = outputs.pooler_output
#             feats = F.normalize(feats, p=2, dim=-1)
#
#             img_embs.append(feats.cpu().numpy())
#
#         # Ekstraksi embedding caption Indonesia
#         for i in range(0, len(cap_id), txt_bs):
#             batch_txt = [str(t) for t in cap_id[i : i + txt_bs]]
#
#             inputs = processor(
#                 text=batch_txt,
#                 padding="max_length",
#                 truncation=True,
#                 return_tensors="pt"
#             )
#
#             outputs = model.get_text_features(**inputs)
#             feats = outputs.pooler_output
#             feats = F.normalize(feats, p=2, dim=-1)
#
#             ti_embs.append(feats.cpu().numpy())
#
#         # Ekstraksi embedding caption Inggris
#         for i in range(0, len(cap_en), txt_bs):
#             batch_txt = [str(t) for t in cap_en[i : i + txt_bs]]
#
#             inputs = processor(
#                 text=batch_txt,
#                 padding="max_length",
#                 truncation=True,
#                 return_tensors="pt"
#             )
#
#             outputs = model.get_text_features(**inputs)
#             feats = outputs.pooler_output
#             feats = F.normalize(feats, p=2, dim=-1)
#
#             te_embs.append(feats.cpu().numpy())
#
#     del model, processor
#     gc.collect()
#
#     return np.vstack(img_embs), np.vstack(ti_embs), np.vstack(te_embs)
#
#
# # Ekstraksi untuk Model Berbasis OpenCLIP
# def encode_openclip_model(arch_name, pretrained_name, imgs, cap_id, cap_en, img_bs=32, txt_bs=64, lang_id=None, lang_en=None):
#     """
#     Ekstraksi untuk:
#     - Varian 1: laion/CLIP-ViT-B-32-xlm-roberta-base-laion5B-s13B-b90k
#     - Varian 3: visheratin/nllb-clip-base-siglip (dengan kode bahasa NLLB-200: ind_Latn & eng_Latn)
#     """
#     model, _, preprocess = open_clip.create_model_and_transforms(arch_name, pretrained=pretrained_name, device="cpu")
#     tokenizer = open_clip.get_tokenizer(arch_name)
#     model.eval()
#     model.requires_grad_(False)  # Untuk Memastikan seluruh bobot pretrained tetap frozen
#     
#     img_embs, ti_embs, te_embs = [], [], []
#     with torch.no_grad():
#         for i in range(0, len(imgs), img_bs):
#             batch_tensors = torch.stack([preprocess(im) for im in imgs[i : i + img_bs]])
#             feats = F.normalize(model.encode_image(batch_tensors), p=2, dim=-1)
#             img_embs.append(feats.cpu().numpy())
#
#         for i in range(0, len(cap_id), txt_bs):
#             batch_txt = [str(t) for t in cap_id[i : i + txt_bs]]
#             
#             if lang_id:
#                 tokenizer.set_language(lang_id)
#             tokens = tokenizer(batch_txt) # memberitahu tokenizer bahasa mana yang sedang dibaca
#             
#             feats = F.normalize(model.encode_text(tokens), p=2, dim=-1)
#             ti_embs.append(feats.cpu().numpy())
#
#         for i in range(0, len(cap_en), txt_bs):
#             batch_txt = [str(t) for t in cap_en[i : i + txt_bs]]
#
#             if lang_en:
#                 tokenizer.set_language(lang_en)
#             tokens = tokenizer(batch_txt) # memberitahu tokenizer bahasa mana yang sedang dibaca
#
#             feats = F.normalize(model.encode_text(tokens), p=2, dim=-1)
#             te_embs.append(feats.cpu().numpy())
#
#     del model, preprocess, tokenizer
#     gc.collect()
#     return np.vstack(img_embs), np.vstack(ti_embs), np.vstack(te_embs)
#
# # Manajer Utama & Pengatur Cache
# def run_encoding(variant_key, imgs, cap_id, cap_en, use_cache=True):
#     """Menjalankan ekstraksi sesuai 4 model rencana kita + simpan cache"""
#     cache_file = os.path.join(PATH_CACHE, f"{variant_key}.pt")
#     t0 = time.time()
#
#     if use_cache and os.path.exists(cache_file):
#         data = torch.load(cache_file, map_location="cpu", weights_only=False)
#         print(f"[{variant_key}] Loaded dari cache ({data['elapsed_sec']} dtk)")
#         return data["E"], data["Ti"], data["Te"], data["elapsed_sec"]
#
#     print(f"[{variant_key}] Mengekstrak embedding di CPU...")
#
#     if variant_key == "baseline_m1":
#         # 1. Baseline M1 (100% kode asli Peran 2)
#         img_model = SentenceTransformer("sentence-transformers/clip-ViT-B-32")
#         txt_model = SentenceTransformer("sentence-transformers/clip-ViT-B-32-multilingual-v1")
#         E  = img_model.encode(imgs,   batch_size=32, normalize_embeddings=True)
#         Ti = txt_model.encode(cap_id, batch_size=64, normalize_embeddings=True)
#         Te = img_model.encode(cap_en, batch_size=64, normalize_embeddings=True)
#         del img_model, txt_model
#         gc.collect()
#
#     elif variant_key == "varian1_laion5b":
#         # 2. Varian 1: LAION-5B XLM-RoBERTa-Base ViT-B-32
#         E, Ti, Te = encode_openclip_model(
#             arch_name="xlm-roberta-base-ViT-B-32",
#             pretrained_name="laion5b_s13b_b90k",
#             imgs=imgs, cap_id=cap_id, cap_en=cap_en
#         )
#
#     elif variant_key == "varian2_siglip2":
#         # 3. Varian 2: Google SigLIP 2 Base Patch16-224
#         E, Ti, Te = encode_siglip2_hf(
#             model_id="google/siglip2-base-patch16-224",
#             imgs=imgs, cap_id=cap_id, cap_en=cap_en
#         )
#
#     elif variant_key == "varian3_nllb_clip":
#         # 4. Varian 3: NLLB-CLIP Base SigLIP (visheratin/nllb-clip-base-siglip)
#         E, Ti, Te = encode_openclip_model(
#             arch_name="nllb-clip-base-siglip",
#             pretrained_name="v1",
#             imgs=imgs, cap_id=cap_id, cap_en=cap_en,
#             lang_id="ind_Latn", lang_en="eng_Latn"
#         )
#
#     else:
#         raise ValueError(f"Varian {variant_key} tidak dikenal.")
#
#     elapsed_sec = round(time.time() - t0, 2) # Hitung Waktu & Simpan Cache
#     torch.save({"E": E, "Ti": Ti, "Te": Te, "elapsed_sec": elapsed_sec}, cache_file)
#     print(f"[{variant_key}] Selesai dalam {elapsed_sec} dtk | Shape: E={E.shape}, Ti={Ti.shape}, Te={Te.shape}")
#     return E, Ti, Te, elapsed_sec
#
# print("Fungsi ekstraksi ke-4 model siap!")


### LAION model encoder baru

In [18]:
# ============================================================
# [REVISI 2026-11-25] MODEL ENCODER LAION-ONLY
# Kode lama multi-model ada tepat di sel sebelumnya dalam bentuk komentar.
# ============================================================

LAION_VARIANT_KEY = "varian1_laion5b"
LAION_ARCH_NAME = "xlm-roberta-base-ViT-B-32"
LAION_PRETRAINED_NAME = "laion5b_s13b_b90k"
LAION_CHECKPOINT = "laion/CLIP-ViT-B-32-xlm-roberta-base-laion5B-s13B-b90k"


def encode_laion_only(images, captions_id, captions_en, img_bs=32, txt_bs=64):
    """Ekstraksi embedding gambar, caption Indonesia, dan caption Inggris dengan LAION.

    Konfigurasi CPU dipertahankan sesuai implementasi awal notebook agar perubahan
    eksperimen hanya membatasi model, bukan sekaligus mengubah perangkat komputasi.
    """
    if not (len(images) == len(captions_id) == len(captions_en)):
        raise ValueError("Jumlah gambar, caption Indonesia, dan caption Inggris harus sama.")
    if len(images) == 0:
        raise ValueError("Dataset kosong; tidak ada sampel untuk diencode.")
    if img_bs < 1 or txt_bs < 1:
        raise ValueError("Ukuran batch harus minimal 1.")

    t0 = time.time()
    model = tokenizer = preprocess = None
    try:
        model, _, preprocess = open_clip.create_model_and_transforms(
            LAION_ARCH_NAME,
            pretrained=LAION_PRETRAINED_NAME,
            device="cpu"
        )
        tokenizer = open_clip.get_tokenizer(LAION_ARCH_NAME)
        model.eval()
        model.requires_grad_(False)

        total_parameters = sum(p.numel() for p in model.parameters())
        vision_parameters = sum(p.numel() for p in model.visual.parameters())
        non_vision_parameters = total_parameters - vision_parameters
        trainable_parameters = sum(
            p.numel() for p in model.parameters() if p.requires_grad
        )

        image_embeddings, caption_id_embeddings, caption_en_embeddings = [], [], []

        with torch.no_grad():
            # Embedding gambar.
            for start in range(0, len(images), img_bs):
                batch_images = images[start:start + img_bs]
                image_tensor = torch.stack([preprocess(im) for im in batch_images])
                features = model.encode_image(image_tensor)
                features = F.normalize(features.float(), p=2, dim=-1)
                image_embeddings.append(features.cpu().numpy())

            # Embedding caption Bahasa Indonesia.
            for start in range(0, len(captions_id), txt_bs):
                batch_text = [str(t) for t in captions_id[start:start + txt_bs]]
                tokens = tokenizer(batch_text)
                features = model.encode_text(tokens)
                features = F.normalize(features.float(), p=2, dim=-1)
                caption_id_embeddings.append(features.cpu().numpy())

            # Embedding caption Bahasa Inggris.
            for start in range(0, len(captions_en), txt_bs):
                batch_text = [str(t) for t in captions_en[start:start + txt_bs]]
                tokens = tokenizer(batch_text)
                features = model.encode_text(tokens)
                features = F.normalize(features.float(), p=2, dim=-1)
                caption_en_embeddings.append(features.cpu().numpy())

        E_laion = np.vstack(image_embeddings).astype(np.float32, copy=False)
        Ti_laion = np.vstack(caption_id_embeddings).astype(np.float32, copy=False)
        Te_laion = np.vstack(caption_en_embeddings).astype(np.float32, copy=False)

        model_stats = {
            "total_parameters": int(total_parameters),
            "vision_parameters": int(vision_parameters),
            "non_vision_parameters": int(non_vision_parameters),
            "trainable_parameters": int(trainable_parameters)
        }
        elapsed_sec = round(time.time() - t0, 2)
        return E_laion, Ti_laion, Te_laion, elapsed_sec, model_stats

    finally:
        if model is not None:
            del model
        if tokenizer is not None:
            del tokenizer
        if preprocess is not None:
            del preprocess
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


print(f"Encoder LAION siap: {LAION_CHECKPOINT}")


Encoder LAION siap: laion/CLIP-ViT-B-32-xlm-roberta-base-laion5B-s13B-b90k


# Smoke test — LAION-only

Sebelum ekstraksi penuh, model LAION diuji menggunakan maksimal dua sampel untuk memastikan embedding gambar dan caption Indonesia/Inggris terbentuk, dimensinya konsisten, nilainya valid, dan ternormalisasi.


In [19]:
# [ARSIP NONAKTIF: Smoke test empat model sebelumnya; diganti oleh smoke test LAION-only berikutnya.]

# # Smoke test untuk memastikan keempat encoder dapat menghasilkan embedding yang valid.
# # Pengujian memakai dua sampel dan tidak menggunakan atau membuat cache eksperimen.
#
# N_SMOKE = 2 # Mengambil 2 gambar dan 2 pasangan caption (ID & EN) pertama sebagai bahan uji coba.
# smoke_imgs = imgs[:N_SMOKE]
# smoke_cap_id = cap_id[:N_SMOKE]
# smoke_cap_en = cap_en[:N_SMOKE]
#
# assert len(smoke_imgs) == N_SMOKE
# assert len(smoke_cap_id) == N_SMOKE
# assert len(smoke_cap_en) == N_SMOKE
#
# # cek hasil smoke, mencetak status LOLOS kalau sesuai requirement
# def check_smoke_result(variant_key, E_test, Ti_test, Te_test, elapsed):
#     """Memeriksa jumlah data, dimensi, nilai numerik, dan normalisasi embedding."""
#     embeddings = {
#         "Image": E_test,
#         "Caption ID": Ti_test,
#         "Caption EN": Te_test
#     }
#
#     assert E_test.shape[0] == N_SMOKE
#     assert Ti_test.shape[0] == N_SMOKE
#     assert Te_test.shape[0] == N_SMOKE
#     assert E_test.shape[1] == Ti_test.shape[1] == Te_test.shape[1]
#
#     for name, values in embeddings.items():
#         assert np.isfinite(values).all(), (
#             f"{variant_key}: ditemukan NaN/Inf pada {name}."
#         )
#         norms = np.linalg.norm(values, axis=1)
#         assert np.allclose(norms, 1.0, atol=1e-3), (
#             f"{variant_key}: embedding {name} belum ternormalisasi."
#         )
#
#     print(
#         f"{variant_key:20s} | "
#         f"dimensi={E_test.shape[1]:4d} | "
#         f"waktu={elapsed:8.2f} detik | LOLOS"
#     )
#
# # Menguji keempat varian satu per satu tanpa menyimpan file cache
# # Setelah setiap model diuji, variabelnya langsung dihapus (del ... dan gc.collect()) agar RAM tidak penuh.
# def smoke_test_all_variants():
#     """Menjalankan smoke test satu per satu agar penggunaan memori tetap terkendali."""
#     results = []
#
#     print("Smoke test menggunakan 2 gambar dan 2 caption per bahasa.")
#     print("-" * 78)
#
#     smoke_variants = [
#         "baseline_m1", "varian1_laion5b",
#         "varian2_siglip2", "varian3_nllb_clip"
#     ]
#     for variant_key in smoke_variants:
#         t0 = time.time()
#
#         if variant_key == "baseline_m1":
#             image_encoder = SentenceTransformer(
#                 "sentence-transformers/clip-ViT-B-32"
#             )
#             text_encoder = SentenceTransformer(
#                 "sentence-transformers/clip-ViT-B-32-multilingual-v1"
#             )
#             image_encoder.eval()
#             text_encoder.eval()
#
#             with torch.no_grad():
#                 E_test = image_encoder.encode(
#                     smoke_imgs, batch_size=2,
#                     normalize_embeddings=True, convert_to_numpy=True,
#                     show_progress_bar=False
#                 )
#                 Ti_test = text_encoder.encode(
#                     smoke_cap_id, batch_size=2,
#                     normalize_embeddings=True, convert_to_numpy=True,
#                     show_progress_bar=False
#                 )
#                 Te_test = image_encoder.encode(
#                     smoke_cap_en, batch_size=2,
#                     normalize_embeddings=True, convert_to_numpy=True,
#                     show_progress_bar=False
#                 )
#
#             del image_encoder, text_encoder
#             gc.collect()
#
#         elif variant_key == "varian1_laion5b":
#             E_test, Ti_test, Te_test = encode_openclip_model(
#                 arch_name="xlm-roberta-base-ViT-B-32",
#                 pretrained_name="laion5b_s13b_b90k",
#                 imgs=smoke_imgs,
#                 cap_id=smoke_cap_id,
#                 cap_en=smoke_cap_en,
#                 img_bs=2,
#                 txt_bs=2
#             )
#
#         elif variant_key == "varian2_siglip2":
#             E_test, Ti_test, Te_test = encode_siglip2_hf(
#                 model_id="google/siglip2-base-patch16-224",
#                 imgs=smoke_imgs,
#                 cap_id=smoke_cap_id,
#                 cap_en=smoke_cap_en,
#                 img_bs=2,
#                 txt_bs=2
#             )
#
#         elif variant_key == "varian3_nllb_clip":
#             E_test, Ti_test, Te_test = encode_openclip_model(
#                 arch_name="nllb-clip-base-siglip",
#                 pretrained_name="v1",
#                 imgs=smoke_imgs,
#                 cap_id=smoke_cap_id,
#                 cap_en=smoke_cap_en,
#                 img_bs=2,
#                 txt_bs=2,
#                 lang_id="ind_Latn",
#                 lang_en="eng_Latn"
#             )
#
#         elapsed = time.time() - t0
#         check_smoke_result(
#             variant_key, E_test, Ti_test, Te_test, elapsed
#         )
#
#         results.append({
#             "varian": variant_key,
#             "dimensi_embedding": E_test.shape[1],
#             "waktu_smoke_test_detik": round(elapsed, 2),
#             "status": "LOLOS"
#         })
#
#         del E_test, Ti_test, Te_test
#         gc.collect()
#
#     return pd.DataFrame(results)
#
#
# smoke_test_results = smoke_test_all_variants()
# display(smoke_test_results)


### LAION smoke test baru

In [20]:
# ============================================================
# [REVISI 2026-11-25] SMOKE TEST LAION-ONLY
# ============================================================

N_SMOKE = min(2, len(imgs))
if N_SMOKE == 0:
    raise ValueError("Tidak ada gambar pada dataset untuk smoke test.")

smoke_imgs = imgs[:N_SMOKE]
smoke_cap_id = cap_id[:N_SMOKE]
smoke_cap_en = cap_en[:N_SMOKE]

E_smoke, Ti_smoke, Te_smoke, smoke_elapsed, smoke_model_stats = encode_laion_only(
    smoke_imgs,
    smoke_cap_id,
    smoke_cap_en,
    img_bs=N_SMOKE,
    txt_bs=N_SMOKE
)

assert E_smoke.shape[0] == Ti_smoke.shape[0] == Te_smoke.shape[0] == N_SMOKE
assert E_smoke.shape[1] == Ti_smoke.shape[1] == Te_smoke.shape[1]

for embedding_name, embedding_values in [
    ("Image", E_smoke),
    ("Caption ID", Ti_smoke),
    ("Caption EN", Te_smoke)
]:
    assert np.isfinite(embedding_values).all(), f"{embedding_name}: NaN/Inf ditemukan."
    norms = np.linalg.norm(embedding_values, axis=1)
    assert np.allclose(norms, 1.0, atol=1e-3), f"{embedding_name}: embedding belum ternormalisasi."

print("SMOKE TEST LAION: LOLOS")
print(f"Jumlah sampel : {N_SMOKE}")
print(f"Dimensi       : {E_smoke.shape[1]}")
print(f"Waktu         : {smoke_elapsed:.2f} detik")
print(f"Total params  : {smoke_model_stats['total_parameters']:,}")

del E_smoke, Ti_smoke, Te_smoke
_ = gc.collect()


open_clip_pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.46GB            

open_clip_pytorch_model.bin: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

SMOKE TEST LAION: LOLOS
Jumlah sampel : 2
Dimensi       : 512
Waktu         : 25.93 detik
Total params  : 366,121,473


# Ekstraksi embedding penuh — LAION-only

Setelah smoke test berhasil, model LAION dijalankan pada seluruh data validation. Hasil embedding disiapkan sebagai input evaluasi Recall@K, mAP@K, dan uji kasus sulit.


In [21]:
# [ARSIP NONAKTIF: Ekstraksi seluruh varian sebelumnya; diganti oleh ekstraksi LAION-only berikutnya.]

# # Ekstraksi embedding untuk seluruh varian pada data validation.
# # use_cache=False memastikan hasil dihitung ulang, bukan diambil dari cache sebelumnya.
#
# DAFTAR_VARIAN = [
#     "baseline_m1",
#     "varian1_laion5b",
#     "varian2_siglip2",
#     "varian3_nllb_clip"
# ]
#
# embeddings_dict = {}
# ringkasan_ekstraksi = []
# jumlah_data = len(imgs)
#
# assert jumlah_data == len(cap_id) == len(cap_en), (
#     "Jumlah gambar dan caption tidak sama. Periksa kembali data validation."
# ) # Memastikan jumlah gambar dan caption id en sama
#
# for variant_key in DAFTAR_VARIAN:
#     print(f"\nMemproses {variant_key} ({jumlah_data} sampel)...")
#
#     E_v, Ti_v, Te_v, waktu_v = run_encoding(
#         variant_key,
#         imgs,
#         cap_id,
#         cap_en,
#         use_cache=False # memaksa program menghitung ulang embedding dari nol
#     )
#
#     # Memastikan setiap embedding tetap sejajar dengan baris dataset.
#     assert E_v.shape[0] == jumlah_data
#     assert Ti_v.shape[0] == jumlah_data
#     assert Te_v.shape[0] == jumlah_data
#     assert E_v.shape[1] == Ti_v.shape[1] == Te_v.shape[1]
#
#     # Memeriksa nilai embedding sebelum diserahkan ke tahap evaluasi.
#     for nama, arr in [("Image", E_v), ("Caption ID", Ti_v), ("Caption EN", Te_v)]:
#         assert np.isfinite(arr).all(), (
#             f"{variant_key}: ditemukan NaN/Inf pada {nama}."
#         )
#         norma = np.linalg.norm(arr, axis=1)
#         assert np.allclose(norma, 1.0, atol=1e-3), (
#             f"{variant_key}: embedding {nama} belum ternormalisasi."
#         )
#
#     embeddings_dict[variant_key] = {
#         "E": E_v,
#         "Ti": Ti_v,
#         "Te": Te_v,
#         "waktu_jalan_detik": waktu_v
#     }
#
#     ringkasan_ekstraksi.append({
#         "varian": variant_key,
#         "jumlah_data": jumlah_data,
#         "dimensi_embedding": int(E_v.shape[1]),
#         "waktu_ekstraksi_detik": waktu_v,
#         "status": "BERHASIL"
#     })
#
#
# ringkasan_ekstraksi = pd.DataFrame(ringkasan_ekstraksi)
# display(ringkasan_ekstraksi)
#
#
# # Menetapkan baseline M1 sebagai referensi untuk sel evaluasi berikutnya (17-22).
# # Seluruh embedding empat varian tetap tersimpan di embeddings_dict.
# E  = embeddings_dict["baseline_m1"]["E"]
# Ti = embeddings_dict["baseline_m1"]["Ti"]
# Te = embeddings_dict["baseline_m1"]["Te"]
#
# print("Embedding baseline M1 telah disiapkan untuk evaluasi Recall@K dan mAP@K.")


### LAION full extraction baru

In [22]:
# ============================================================
# [REVISI 2026-11-25] EKSTRAKSI PENUH LAION-ONLY
# ============================================================

DAFTAR_VARIAN = [LAION_VARIANT_KEY]  # hanya LAION yang aktif
EXPORT_DIR_LAION = "/kaggle/working/LAION_Only_Experiment_Results"
PATH_CACHE_LAION = "/kaggle/working/cache_laion_only/"
os.makedirs(EXPORT_DIR_LAION, exist_ok=True)
os.makedirs(PATH_CACHE_LAION, exist_ok=True)

jumlah_data = len(imgs)
assert jumlah_data == len(cap_id) == len(cap_en), (
    "Jumlah gambar dan caption tidak sama. Periksa kembali data validation."
)

# False = hitung ulang dari model; ubah ke True hanya bila ingin memakai cache LAION.
USE_CACHE_LAION = False
LAION_CACHE_FILE = os.path.join(PATH_CACHE_LAION, "laion5b_validation_embeddings.pt")

if USE_CACHE_LAION and os.path.exists(LAION_CACHE_FILE):
    print("Memuat embedding LAION dari cache:", LAION_CACHE_FILE)
    cached = torch.load(LAION_CACHE_FILE, map_location="cpu", weights_only=False)
    E = cached["E"]
    Ti = cached["Ti"]
    Te = cached["Te"]
    waktu_laion = float(cached["elapsed_sec"])
    LAION_MODEL_STATS = cached["model_stats"]
else:
    print(f"Menjalankan LAION pada {jumlah_data} sampel validation...")
    E, Ti, Te, waktu_laion, LAION_MODEL_STATS = encode_laion_only(
        imgs,
        cap_id,
        cap_en,
        img_bs=32,
        txt_bs=64
    )
    torch.save({
        "E": E,
        "Ti": Ti,
        "Te": Te,
        "elapsed_sec": waktu_laion,
        "model_stats": LAION_MODEL_STATS,
        "checkpoint": LAION_CHECKPOINT
    }, LAION_CACHE_FILE)

# Pemeriksaan hasil ekstraksi penuh.
assert E.shape[0] == Ti.shape[0] == Te.shape[0] == jumlah_data
assert E.shape[1] == Ti.shape[1] == Te.shape[1]
for embedding_name, embedding_values in [("Image", E), ("Caption ID", Ti), ("Caption EN", Te)]:
    assert np.isfinite(embedding_values).all(), f"{embedding_name}: ditemukan NaN/Inf."
    assert np.allclose(
        np.linalg.norm(embedding_values, axis=1), 1.0, atol=1e-3
    ), f"{embedding_name}: embedding belum L2-normalized."

embeddings_dict = {
    LAION_VARIANT_KEY: {
        "E": E,
        "Ti": Ti,
        "Te": Te,
        "waktu_jalan_detik": waktu_laion,
        "model_stats": LAION_MODEL_STATS
    }
}

ringkasan_ekstraksi = pd.DataFrame([{
    "varian": LAION_VARIANT_KEY,
    "checkpoint": LAION_CHECKPOINT,
    "jumlah_data": jumlah_data,
    "dimensi_embedding": int(E.shape[1]),
    "waktu_ekstraksi_detik": waktu_laion,
    "status": "BERHASIL"
}])

display(ringkasan_ekstraksi)
print("Embedding LAION siap untuk evaluasi retrieval dan M3.")
print(f"Cache : {LAION_CACHE_FILE}")


Menjalankan LAION pada 275 sampel validation...


,varian,checkpoint,jumlah_data,dimensi_embedding,waktu_ekstraksi_detik,status
0,varian1_laion5b,laion/CLIP-ViT-B-32-xlm-roberta-base-laion5B-s...,275,512,98.71,BERHASIL


Embedding LAION siap untuk evaluasi retrieval dan M3.
Cache : /kaggle/working/cache_laion_only/laion5b_validation_embeddings.pt


In [23]:
# [ARSIP NONAKTIF: Penyimpanan multi-model sebelumnya; diganti oleh penyimpanan LAION-only berikutnya.]

# # Menyimpan embedding dan metadata agar dapat digunakan oleh Peran 4 yaitu evaluator dan uji sulit 
# import shutil
# from pathlib import Path
#
# if "EXPORT_DIR" not in globals() or not EXPORT_DIR:
#     EXPORT_DIR = "/kaggle/working/M2_Experiment_Results"
# os.makedirs(EXPORT_DIR, exist_ok=True) 
#
# image_ids = df_new["image_id"].astype(str).tolist()
# assert len(image_ids) == len(imgs) # Mengambil daftar image_id secara berurutan dan memastikan jumlahnya sama persis dengan jumlah gambar yang diekstrak 
#
# MODEL_INFO = {
#     "baseline_m1": {
#         "checkpoint": "sentence-transformers/clip-ViT-B-32 + sentence-transformers/clip-ViT-B-32-multilingual-v1",
#         "backend": "SentenceTransformers"
#     },
#     "varian1_laion5b": {
#         "checkpoint": "laion/CLIP-ViT-B-32-xlm-roberta-base-laion5B-s13B-b90k",
#         "backend": "OpenCLIP"
#     },
#     "varian2_siglip2": {
#         "checkpoint": "google/siglip2-base-patch16-224",
#         "backend": "Transformers"
#     },
#     "varian3_nllb_clip": {
#         "checkpoint": "visheratin/nllb-clip-base-siglip",
#         "backend": "OpenCLIP",
#         "text_language_codes": {"caption_id": "ind_Latn", "caption_en": "eng_Latn"}
#     }
# }
#
# ringkasan_file = []
#
# for variant_key in DAFTAR_VARIAN:
#     data = embeddings_dict[variant_key]
#     folder_varian = os.path.join(EXPORT_DIR, variant_key)
#     os.makedirs(folder_varian, exist_ok=True)
#
#     # Menyimpan tiga jenis embedding sebagai file terpisah.
#     np.save(os.path.join(folder_varian, "E.npy"), np.asarray(data["E"], dtype=np.float32))
#     np.save(os.path.join(folder_varian, "Ti.npy"), np.asarray(data["Ti"], dtype=np.float32))
#     np.save(os.path.join(folder_varian, "Te.npy"), np.asarray(data["Te"], dtype=np.float32))
#
#     # inisialisasi metadata.json
#     # berisi nama checkpoint model yang dipakai, dimensi vektor, waktu proses, nilai SEED, image_ids_in_dataset_order
#     metadata_varian = {
#         "variant": variant_key,
#         "model": MODEL_INFO[variant_key],
#         "dataset_file": "val_clean.csv",
#         "split": "validation",
#         "num_samples": len(image_ids),
#         "embedding_dimension": int(data["E"].shape[1]),
#         "embedding_files": {
#             "E": "E.npy (image embeddings)",
#             "Ti": "Ti.npy (Indonesian caption embeddings)",
#             "Te": "Te.npy (English caption embeddings)"
#         },
#         "normalization": "L2-normalized",
#         "pretrained_weights_frozen": True,
#         "elapsed_seconds_including_model_loading": float(data["waktu_jalan_detik"]),
#         "seed": SEED,
#         "row_alignment": "Baris ke-i pada E, Ti, dan Te mengacu pada image_id ke-i di bawah ini.",
#         "image_ids_in_dataset_order": image_ids
#     }
#
#     with open(os.path.join(folder_varian, "metadata.json"), "w", encoding="utf-8") as f:
#         json.dump(metadata_varian, f, indent=2, ensure_ascii=False)
#
#     ringkasan_file.append({
#         "varian": variant_key,
#         "checkpoint": MODEL_INFO[variant_key]["checkpoint"],
#         "jumlah_data": len(image_ids),
#         "dimensi_embedding": int(data["E"].shape[1]),
#         "waktu_ekstraksi_detik": data["waktu_jalan_detik"],
#         "folder_output": variant_key
#     })
#
# summary_df = pd.DataFrame(ringkasan_file) # berisi nama model, dimensi, dan kecepatan ekstraksi
# summary_df.to_csv(os.path.join(EXPORT_DIR, "experiment_summary.csv"), index=False)
#
# # Membuat satu arsip agar seluruh hasil mudah diunduh dari Kaggle.
# ZIP_PATH = shutil.make_archive(EXPORT_DIR, "zip", root_dir=EXPORT_DIR)
#
# print("Semua embedding dan metadata berhasil disimpan.")
# print(f"Folder hasil : {EXPORT_DIR}")
# print(f"Arsip ZIP    : {ZIP_PATH}")
# display(summary_df)


### LAION save metadata baru

In [24]:
# ============================================================
# [REVISI 2026-11-25] SIMPAN EMBEDDING DAN METADATA LAION-ONLY
# ============================================================

import shutil
from pathlib import Path

image_ids = df_new["image_id"].astype(str).tolist()
assert len(image_ids) == len(E), "Jumlah image_id harus sama dengan embedding gambar."

laion_folder = os.path.join(EXPORT_DIR_LAION, LAION_VARIANT_KEY)
os.makedirs(laion_folder, exist_ok=True)

np.save(os.path.join(laion_folder, "E.npy"), np.asarray(E, dtype=np.float32))
np.save(os.path.join(laion_folder, "Ti.npy"), np.asarray(Ti, dtype=np.float32))
np.save(os.path.join(laion_folder, "Te.npy"), np.asarray(Te, dtype=np.float32))

metadata_laion = {
    "variant": LAION_VARIANT_KEY,
    "model": {
        "checkpoint": LAION_CHECKPOINT,
        "architecture": LAION_ARCH_NAME,
        "pretrained_tag": LAION_PRETRAINED_NAME,
        "backend": "OpenCLIP"
    },
    "dataset_file": "val_clean.csv",
    "split": "validation",
    "num_samples": len(image_ids),
    "embedding_dimension": int(E.shape[1]),
    "embedding_files": {
        "E": "E.npy (image embeddings)",
        "Ti": "Ti.npy (Indonesian caption embeddings)",
        "Te": "Te.npy (English caption embeddings)"
    },
    "normalization": "L2-normalized",
    "pretrained_weights_frozen": True,
    "elapsed_seconds_including_model_loading": float(waktu_laion),
    "model_parameters": LAION_MODEL_STATS,
    "seed": SEED,
    "device": "cpu",
    "row_alignment": "Baris ke-i pada E, Ti, dan Te mengacu pada image_id ke-i di bawah ini.",
    "image_ids_in_dataset_order": image_ids
}

with open(os.path.join(laion_folder, "metadata.json"), "w", encoding="utf-8") as f:
    json.dump(metadata_laion, f, indent=2, ensure_ascii=False)

summary_df = pd.DataFrame([{
    "varian": LAION_VARIANT_KEY,
    "checkpoint": LAION_CHECKPOINT,
    "jumlah_data": len(image_ids),
    "dimensi_embedding": int(E.shape[1]),
    "waktu_ekstraksi_detik": waktu_laion,
    "folder_output": LAION_VARIANT_KEY
}])
summary_df.to_csv(
    os.path.join(EXPORT_DIR_LAION, "experiment_summary.csv"),
    index=False
)

laion_zip_path = EXPORT_DIR_LAION + ".zip"
if os.path.exists(laion_zip_path):
    os.remove(laion_zip_path)
ZIP_PATH = shutil.make_archive(
    EXPORT_DIR_LAION,
    "zip",
    root_dir=EXPORT_DIR_LAION
)

print("Embedding dan metadata LAION berhasil disimpan.")
print(f"Folder hasil : {EXPORT_DIR_LAION}")
print(f"Arsip ZIP    : {ZIP_PATH}")
display(summary_df)


Embedding dan metadata LAION berhasil disimpan.
Folder hasil : /kaggle/working/LAION_Only_Experiment_Results
Arsip ZIP    : /kaggle/working/LAION_Only_Experiment_Results.zip


,varian,checkpoint,jumlah_data,dimensi_embedding,waktu_ekstraksi_detik,folder_output
0,varian1_laion5b,laion/CLIP-ViT-B-32-xlm-roberta-base-laion5B-s...,275,512,98.71,varian1_laion5b


In [25]:
# [ARSIP NONAKTIF: Verifikasi empat embedding model sebelumnya; diganti oleh verifikasi LAION-only berikutnya.]

# # mengecek lagi hasil embedding yg di compress
# if "ZIP_PATH" not in globals() or not os.path.exists(ZIP_PATH):
#     ZIP_PATH = f"{EXPORT_DIR}.zip" if "EXPORT_DIR" in globals() else "/kaggle/working/M2_Experiment_Results.zip"
#
# dimensi_model = {
#     "baseline_m1": 512,
#     "varian1_laion5b": 512,
#     "varian2_siglip2": 768,
#     "varian3_nllb_clip": 768
# }
#
# with zipfile.ZipFile(ZIP_PATH, "r") as z:
#     assert z.testzip() is None, "Arsip ZIP rusak."
#
#     nama_file = set(z.namelist())
#     assert "experiment_summary.csv" in nama_file
#
#     for variant, dim in dimensi_model.items():
#         print(f"\nMemeriksa {variant}...")
#
#         # Memastikan seluruh file tersedia.
#         files = ["E.npy", "Ti.npy", "Te.npy", "metadata.json"]
#         assert all(f"{variant}/{f}" in nama_file for f in files)
#
#         # Memeriksa metadata.
#         metadata = json.loads(z.read(f"{variant}/metadata.json"))
#         n_samples_expected = len(image_ids) if "image_ids" in globals() else 275
#         assert metadata["num_samples"] == n_samples_expected
#         assert metadata["embedding_dimension"] == dim
#         assert len(metadata["image_ids_in_dataset_order"]) == n_samples_expected
#
#         # Memeriksa embedding.
#         for file in ["E.npy", "Ti.npy", "Te.npy"]:
#             arr = np.load(BytesIO(z.read(f"{variant}/{file}")))
#
#             assert arr.shape == (n_samples_expected, dim)
#             assert np.isfinite(arr).all()
#             assert np.allclose(
#                 np.linalg.norm(arr, axis=1), 1.0, atol=1e-3
#             )
#
#             print(f"{file}: {arr.shape} | VALID")
#
#         print("Metadata: VALID")
#
#     # Memeriksa ringkasan eksperimen.
#     summary = pd.read_csv(BytesIO(z.read("experiment_summary.csv")))
#     assert len(summary) == len(dimensi_model)
#
#     print("\nSeluruh arsip dan embedding berhasil diverifikasi.")
#     display(summary)


### LAION archive verification baru

In [26]:
# ============================================================
# [REVISI 2026-11-25] VERIFIKASI ARSIP LAION-ONLY
# ============================================================

if "ZIP_PATH" not in globals() or not os.path.exists(ZIP_PATH):
    ZIP_PATH = EXPORT_DIR_LAION + ".zip"

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    assert z.testzip() is None, "Arsip ZIP rusak."
    names = set(z.namelist())

    required_files = [
        f"{LAION_VARIANT_KEY}/E.npy",
        f"{LAION_VARIANT_KEY}/Ti.npy",
        f"{LAION_VARIANT_KEY}/Te.npy",
        f"{LAION_VARIANT_KEY}/metadata.json",
        "experiment_summary.csv"
    ]
    missing = [name for name in required_files if name not in names]
    assert not missing, f"File berikut tidak ditemukan di ZIP: {missing}"

    metadata = json.loads(z.read(f"{LAION_VARIANT_KEY}/metadata.json"))
    expected_n = len(image_ids)
    expected_dim = int(metadata["embedding_dimension"])
    assert metadata["num_samples"] == expected_n
    assert len(metadata["image_ids_in_dataset_order"]) == expected_n

    for file_name in ["E.npy", "Ti.npy", "Te.npy"]:
        arr = np.load(BytesIO(z.read(f"{LAION_VARIANT_KEY}/{file_name}")))
        assert arr.shape == (expected_n, expected_dim), (
            f"Shape {file_name} tidak sesuai: {arr.shape}"
        )
        assert np.isfinite(arr).all(), f"{file_name} mengandung NaN/Inf."
        assert np.allclose(np.linalg.norm(arr, axis=1), 1.0, atol=1e-3), (
            f"{file_name} belum L2-normalized."
        )
        print(f"{file_name}: {arr.shape} | VALID")

    summary_check = pd.read_csv(BytesIO(z.read("experiment_summary.csv")))
    assert len(summary_check) == 1
    assert summary_check.iloc[0]["varian"] == LAION_VARIANT_KEY

print("Arsip, embedding, dan metadata LAION berhasil diverifikasi.")
display(summary_check)


E.npy: (275, 512) | VALID
Ti.npy: (275, 512) | VALID
Te.npy: (275, 512) | VALID
Arsip, embedding, dan metadata LAION berhasil diverifikasi.


,varian,checkpoint,jumlah_data,dimensi_embedding,waktu_ekstraksi_detik,folder_output
0,varian1_laion5b,laion/CLIP-ViT-B-32-xlm-roberta-base-laion5B-s...,275,512,98.71,varian1_laion5b


# Lanjutan Evaluasi dan Analisis — LAION-only

Pada tahap ini, embedding dari **satu model LAION** digunakan untuk mengevaluasi cross-modal retrieval.

**Tugas evaluasi:**
1. Menghitung Recall@1, Recall@5, Recall@10, dan mAP@K.
2. Mengevaluasi empat arah retrieval: gambar → teks Indonesia, gambar → teks Inggris, teks Indonesia → gambar, dan teks Inggris → gambar.
3. Mencatat waktu ekstraksi, jumlah parameter, ukuran embedding, dan throughput model LAION.
4. Melakukan uji kasus sulit (M3) untuk melihat perbedaan negasi, relasi, tindakan, dan atribut budaya.

**Input:** embedding LAION yang dihasilkan pada bagian M2.

**Output:** tabel metrik retrieval, ringkasan biaya komputasi, metadata, serta hasil uji kasus sulit.

**Catatan M3:** siapkan tepat lima gambar dan minimal dua kandidat query per gambar. Isi `correct_label` berdasarkan inspeksi gambar sebelum melihat prediksi model.


In [27]:
# Kesamaan Kosinus
sim_img_capId = E @ Ti.T
sim_img_capEn = E @ Te.T
sim_capEn_capId = Te @ Ti.T

print("Kesamaan Kosinus image dengan caption indonesia:\n", sim_img_capId, '\n')
print("Kesamaan Kosinus image dengan caption english:\n", sim_img_capEn, '\n')
print("Kesamaan Kosinus caption english dengan caption indonesia:\n", sim_capEn_capId, '\n')

Kesamaan Kosinus image dengan caption indonesia:
 [[0.24454719 0.19917107 0.2323105  ... 0.23764347 0.0571425  0.10011497]
 [0.22316048 0.22312069 0.22331673 ... 0.2645397  0.03391186 0.07473917]
 [0.18470049 0.13678437 0.21913855 ... 0.19034979 0.03189945 0.07791443]
 ...
 [0.21128023 0.15446788 0.21983892 ... 0.19191262 0.04341335 0.04728653]
 [0.12416728 0.12037297 0.16602618 ... 0.11318988 0.22435784 0.12448898]
 [0.03594991 0.0569041  0.10067226 ... 0.10354724 0.09244625 0.23164184]] 

Kesamaan Kosinus image dengan caption english:
 [[0.2689171  0.18498847 0.2211147  ... 0.18685085 0.06090164 0.11562786]
 [0.21342269 0.24771896 0.2184337  ... 0.23353224 0.01262463 0.06603083]
 [0.21710359 0.14869119 0.21874109 ... 0.17595464 0.02728027 0.08402692]
 ...
 [0.23707907 0.16207919 0.20644613 ... 0.18157464 0.05359549 0.06197831]
 [0.13244365 0.10829543 0.17943302 ... 0.07185581 0.23240814 0.1148002 ]
 [0.06249914 0.02105975 0.09816861 ... 0.02791053 0.0889798  0.24164112]] 

Kesamaan K

In [28]:
print(sim_img_capId.shape)
print(sim_img_capEn.shape)
print(sim_capEn_capId.shape)

(275, 275)
(275, 275)
(275, 275)


In [29]:
# Metrik Recall@K, K = {1,5,10}
def recall_at_K(A, B, ks=(1, 5, 10)):
    rank = (A @ B.T).argsort(dim=1, descending=True)
    truth = torch.arange(len(A)).unsqueeze(1)
    return {k: float((rank[:, :k] == truth).any(1).float().mean()) for k in ks}

# Metrik mAP@K, K = {1,5,10}
def map_at_K(A, B, ks=(1, 5, 10)):
  rank = (A @ B.T).argsort(dim=1, descending=True)
  truth = torch.arange(len(A), device=A.device).unsqueeze(1)
  results = {}
  for k in ks:
    hits = rank[:, :k] == truth

    # Karena hanya ada 1 jawaban benar per kueri,
    # AP@k = 1 / (posisi_peringkat + 1) jika ketemu, dan 0 jika tidak ketemu.
    # argmax(dim=1) mengambil indeks posisi pertama di mana True muncul.
    pos = hits.int().argmax(dim=1)
    has_hit = hits.any(dim=1)

    ap = torch.where(has_hit, 1.0 / (pos.float() + 1.0), 0.0)
    results[k] = float(ap.mean())
  return results


In [30]:
print("Recall@1/5/10:")
print("gambar -> teks-ID :", recall_at_K(torch.tensor(E), torch.tensor(Ti)))
print("teks-ID -> gambar :", recall_at_K(torch.tensor(Ti), torch.tensor(E)))
print("gambar -> teks-EN :", recall_at_K(torch.tensor(E), torch.tensor(Te)))
print("teks-EN -> gambar :", recall_at_K(torch.tensor(Te), torch.tensor(E)), '\n')

print("mAP@1/5/10:")
print("gambar -> teks-ID :", map_at_K(torch.tensor(E), torch.tensor(Ti)))
print("teks-ID -> gambar :", map_at_K(torch.tensor(Ti), torch.tensor(E)))
print("gambar -> teks-EN :", map_at_K(torch.tensor(E), torch.tensor(Te)))
print("teks-EN -> gambar :", map_at_K(torch.tensor(Te), torch.tensor(E)))

Recall@1/5/10:
gambar -> teks-ID : {1: 0.4872727394104004, 5: 0.7636363506317139, 10: 0.8581818342208862}
teks-ID -> gambar : {1: 0.4254545569419861, 5: 0.6836363673210144, 10: 0.7963636517524719}
gambar -> teks-EN : {1: 0.48363634943962097, 5: 0.7636363506317139, 10: 0.8836363554000854}
teks-EN -> gambar : {1: 0.44363635778427124, 5: 0.7272727489471436, 10: 0.8290908932685852} 

mAP@1/5/10:
gambar -> teks-ID : {1: 0.4872727394104004, 5: 0.5923030376434326, 10: 0.6048355102539062}
teks-ID -> gambar : {1: 0.4254545569419861, 5: 0.5194545388221741, 10: 0.5344574451446533}
gambar -> teks-EN : {1: 0.48363634943962097, 5: 0.5918787717819214, 10: 0.6080000400543213}
teks-EN -> gambar : {1: 0.44363635778427124, 5: 0.5515758395195007, 10: 0.5652279853820801}


In [31]:
# [ARSIP NONAKTIF: Ringkasan biaya komputasi empat model sebelumnya; diganti oleh ringkasan LAION-only berikutnya.]

# # Ringkasan biaya komputasi dan parameter setiap varian M2
#
# end_time = time.time()
# waktu_total_tahap = end_time - start_time
#
# print(
#     f"Total waktu sejak timer dimulai: "
#     f"{waktu_total_tahap:.2f} detik "
#     f"({waktu_total_tahap / 60:.2f} menit)"
# )
#
# # Spesifikasi jumlah parameter arsitektur model (Vision + Text)
# # Trainable parameters = 0 karena semua bobot pretrained dibekukan (requires_grad = False)
# PARAMS_MODEL = {
#     "baseline_m1": {
#         "checkpoint": "sentence-transformers/clip-ViT-B-32 + clip-ViT-B-32-multilingual-v1",
#         "vision_params": 87_849_984,
#         "text_params": 135_082_752,
#         "total_params": 222_932_736,
#         "trainable_params": 0
#     },
#     "varian1_laion5b": {
#         "checkpoint": "laion/CLIP-ViT-B-32-xlm-roberta-base-laion5B-s13B-b90k",
#         "vision_params": 87_849_984,
#         "text_params": 278_043_648,
#         "total_params": 365_893_632,
#         "trainable_params": 0
#     },
#     "varian2_siglip2": {
#         "checkpoint": "google/siglip2-base-patch16-224",
#         "vision_params": 86_523_648,
#         "text_params": 134_510_592,
#         "total_params": 221_034_240,
#         "trainable_params": 0
#     },
#     "varian3_nllb_clip": {
#         "checkpoint": "visheratin/nllb-clip-base-siglip",
#         "vision_params": 86_523_648,
#         "text_params": 270_000_000,
#         "total_params": 356_523_648,
#         "trainable_params": 0
#     }
# }
#
# biaya_komputasi_per_varian = {}
# tabel_biaya = []
#
# for variant_key in DAFTAR_VARIAN:
#     data = embeddings_dict[variant_key]
#     p_info = PARAMS_MODEL[variant_key]
#
#     # Mengambil waktu dan dimensi aktual dari hasil ekstraksi.
#     waktu_ekstraksi = float(data["waktu_jalan_detik"])
#     n_samples = len(data["E"])
#     dim = int(data["E"].shape[1])
#
#     # Memastikan informasi parameter konsisten.
#     total_p = int(p_info["total_params"])
#     trainable_p = int(p_info["trainable_params"])
#
#     assert total_p == (
#         p_info["vision_params"] + p_info["text_params"]
#     ), f"Jumlah parameter {variant_key} tidak konsisten."
#
#     # Seluruh embedding harus memiliki jumlah sampel dan dimensi sama.
#     assert all(
#         len(data[k]) == n_samples and data[k].shape[1] == dim
#         for k in ["E", "Ti", "Te"]
#     )
#
#     # Rata-rata waktu pemrosesan satu sampel (gambar + dua caption).
#     ms_per_sample = waktu_ekstraksi / n_samples * 1000
#
#     # Jumlah sampel validation yang diproses setiap detik.
#     throughput = n_samples / waktu_ekstraksi
#
#     # Ukuran aktual ketiga embedding dalam format float32.
#     ukuran_embedding_mb = sum(
#         np.asarray(data[k], dtype=np.float32).nbytes
#         for k in ["E", "Ti", "Te"]
#     ) / (1024 ** 2)
#
#     pct_trainable = (
#         trainable_p / total_p * 100 if total_p > 0 else 0
#     )
#
#     biaya_komputasi_per_varian[variant_key] = {
#         "checkpoint": p_info["checkpoint"],
#         "waktu_ekstraksi_detik": round(waktu_ekstraksi, 2),
#         "latensi_rata_rata_ms_per_sampel": round(ms_per_sample, 2),
#         "throughput_sampel_per_detik": round(throughput, 2),
#         "total_parameters": total_p,
#         "trainable_parameters": trainable_p,
#         "persentase_trainable": round(pct_trainable, 2),
#         "dimensi_embedding": dim,
#         "ukuran_embedding_mb": round(ukuran_embedding_mb, 3)
#     }
#
#     tabel_biaya.append({
#         "Varian": variant_key,
#         "Total Params (M)": round(total_p / 1e6, 2),
#         "Trainable Params": trainable_p,
#         "Dimensi Embedding": dim,
#         "Waktu Ekstraksi (s)": round(waktu_ekstraksi, 2),
#         "Rata-rata Waktu/Sampel (ms)": round(ms_per_sample, 2),
#         "Throughput (sampel/s)": round(throughput, 2),
#         "Ukuran Embedding (MB)": round(ukuran_embedding_mb, 3)
#     })
#
# # Menampilkan hasil perbandingan biaya setiap model.
# df_biaya_komputasi = pd.DataFrame(tabel_biaya)
#
# print("\n=== RINGKASAN BIAYA KOMPUTASI M2 ===")
# display(df_biaya_komputasi)
#
# # Menyimpan hasil agar dapat digunakan Peran 4.
# biaya_path = os.path.join(
#     EXPORT_DIR, "compute_cost_summary.csv"
# )
#
# df_biaya_komputasi.to_csv(biaya_path, index=False)
#
# # Memperbarui arsip ZIP dengan file ringkasan biaya komputasi.
# zip_file = EXPORT_DIR + ".zip"
#
# if os.path.exists(zip_file):
#     os.remove(zip_file)
#
# ZIP_PATH = shutil.make_archive(
#     EXPORT_DIR, "zip", root_dir=EXPORT_DIR
# )
#
# print("\nRingkasan biaya berhasil disimpan.")
# print(f"CSV : {biaya_path}")
# print(f"ZIP : {ZIP_PATH}")


### LAION compute cost baru

In [32]:
# ============================================================
# [REVISI 2026-11-25] BIAYA KOMPUTASI MODEL LAION
# ============================================================

end_time = time.time()
waktu_total_tahap = end_time - start_time
laion_data = embeddings_dict[LAION_VARIANT_KEY]
model_stats = laion_data["model_stats"]

n_samples = len(laion_data["E"])
embedding_dim = int(laion_data["E"].shape[1])
elapsed = float(laion_data["waktu_jalan_detik"])

ms_per_sample = elapsed / n_samples * 1000 if n_samples else 0.0
throughput = n_samples / elapsed if elapsed > 0 else 0.0
embedding_size_mb = sum(
    np.asarray(laion_data[key], dtype=np.float32).nbytes
    for key in ["E", "Ti", "Te"]
) / (1024 ** 2)

total_params = int(model_stats["total_parameters"])
trainable_params = int(model_stats["trainable_parameters"])
trainable_pct = trainable_params / total_params * 100 if total_params else 0.0

biaya_komputasi_per_varian = {
    LAION_VARIANT_KEY: {
        "checkpoint": LAION_CHECKPOINT,
        "waktu_ekstraksi_detik": round(elapsed, 2),
        "latensi_rata_rata_ms_per_sampel": round(ms_per_sample, 2),
        "throughput_sampel_per_detik": round(throughput, 2),
        "total_parameters": total_params,
        "vision_parameters": int(model_stats["vision_parameters"]),
        "non_vision_parameters": int(model_stats["non_vision_parameters"]),
        "trainable_parameters": trainable_params,
        "persentase_trainable": round(trainable_pct, 4),
        "dimensi_embedding": embedding_dim,
        "ukuran_embedding_mb": round(embedding_size_mb, 3)
    }
}

df_biaya_komputasi = pd.DataFrame([{
    "Varian": LAION_VARIANT_KEY,
    "Checkpoint": LAION_CHECKPOINT,
    "Total Params (M)": round(total_params / 1e6, 2),
    "Trainable Params": trainable_params,
    "Dimensi Embedding": embedding_dim,
    "Waktu Ekstraksi (s)": round(elapsed, 2),
    "Rata-rata Waktu/Sampel (ms)": round(ms_per_sample, 2),
    "Throughput (sampel/s)": round(throughput, 2),
    "Ukuran Embedding (MB)": round(embedding_size_mb, 3)
}])

print(f"Total waktu sejak timer dimulai: {waktu_total_tahap:.2f} detik ({waktu_total_tahap / 60:.2f} menit)")
print("\n=== BIAYA KOMPUTASI LAION ===")
display(df_biaya_komputasi)

biaya_path = os.path.join(EXPORT_DIR_LAION, "compute_cost_summary.csv")
df_biaya_komputasi.to_csv(biaya_path, index=False)

if os.path.exists(EXPORT_DIR_LAION + ".zip"):
    os.remove(EXPORT_DIR_LAION + ".zip")
ZIP_PATH = shutil.make_archive(
    EXPORT_DIR_LAION,
    "zip",
    root_dir=EXPORT_DIR_LAION
)
print(f"CSV biaya : {biaya_path}")
print(f"ZIP       : {ZIP_PATH}")


Total waktu sejak timer dimulai: 152.30 detik (2.54 menit)

=== BIAYA KOMPUTASI LAION ===


,Varian,Checkpoint,Total Params (M),Trainable Params,Dimensi Embedding,Waktu Ekstraksi (s),Rata-rata Waktu/Sampel (ms),Throughput (sampel/s),Ukuran Embedding (MB)
0,varian1_laion5b,laion/CLIP-ViT-B-32-xlm-roberta-base-laion5B-s...,366.12,0,512,98.71,358.95,2.79,1.611


CSV biaya : /kaggle/working/LAION_Only_Experiment_Results/compute_cost_summary.csv
ZIP       : /kaggle/working/LAION_Only_Experiment_Results.zip


In [33]:
# [ARSIP NONAKTIF: Ekspor evaluasi dengan biaya per varian sebelumnya; diganti oleh ekspor LAION-only berikutnya.]

# hasil_eksperimen = {
#     "waktu_jalan_detik": round(end_time - start_time, 2),
#     "biaya_komputasi_dan_parameter_per_varian": biaya_komputasi_per_varian if "biaya_komputasi_per_varian" in globals() else {},
#     "evaluasi_metrik": {
#         "Recall_at_K": {
#             "gambar_ke_teks_ID": recall_at_K(torch.tensor(E), torch.tensor(Ti)),
#             "teks_ID_ke_gambar": recall_at_K(torch.tensor(Ti), torch.tensor(E)),
#             "gambar_ke_teks_EN": recall_at_K(torch.tensor(E), torch.tensor(Te)),
#             "teks_EN_ke_gambar": recall_at_K(torch.tensor(Te), torch.tensor(E))
#         },
#         "mAP_at_K": {
#             "gambar_ke_teks_ID": map_at_K(torch.tensor(E), torch.tensor(Ti)),
#             "teks_ID_ke_gambar": map_at_K(torch.tensor(Ti), torch.tensor(E)),
#             "gambar_ke_teks_EN": map_at_K(torch.tensor(E), torch.tensor(Te)),
#             "teks_EN_ke_gambar": map_at_K(torch.tensor(Te), torch.tensor(E))
#         }
#     }
# }
#
# outputLocation = PATH_OUTPUT
# with open(outputLocation, "w", encoding="utf-8") as f:
#     json.dump(hasil_eksperimen, f, indent=4, ensure_ascii=False)
#
# print(f"Hasil berhasil disimpan di: {outputLocation}")


### LAION metrics export baru

In [34]:
# ============================================================
# [REVISI 2026-11-25] SIMPAN HASIL EVALUASI RETRIEVAL LAION
# ============================================================

E_tensor = torch.as_tensor(E, dtype=torch.float32)
Ti_tensor = torch.as_tensor(Ti, dtype=torch.float32)
Te_tensor = torch.as_tensor(Te, dtype=torch.float32)

hasil_eksperimen = {
    "model": {
        "variant": LAION_VARIANT_KEY,
        "checkpoint": LAION_CHECKPOINT,
        "architecture": LAION_ARCH_NAME,
        "pretrained_tag": LAION_PRETRAINED_NAME
    },
    "waktu_jalan_detik_sejak_timer": round(time.time() - start_time, 2),
    "biaya_komputasi": biaya_komputasi_per_varian[LAION_VARIANT_KEY],
    "evaluasi_metrik": {
        "Recall_at_K": {
            "gambar_ke_teks_ID": recall_at_K(E_tensor, Ti_tensor),
            "teks_ID_ke_gambar": recall_at_K(Ti_tensor, E_tensor),
            "gambar_ke_teks_EN": recall_at_K(E_tensor, Te_tensor),
            "teks_EN_ke_gambar": recall_at_K(Te_tensor, E_tensor)
        },
        "mAP_at_K": {
            "gambar_ke_teks_ID": map_at_K(E_tensor, Ti_tensor),
            "teks_ID_ke_gambar": map_at_K(Ti_tensor, E_tensor),
            "gambar_ke_teks_EN": map_at_K(E_tensor, Te_tensor),
            "teks_EN_ke_gambar": map_at_K(Te_tensor, E_tensor)
        }
    }
}

PATH_OUTPUT_LAION = os.path.join(EXPORT_DIR_LAION, "laion_experiment_results.json")
with open(PATH_OUTPUT_LAION, "w", encoding="utf-8") as f:
    json.dump(hasil_eksperimen, f, indent=4, ensure_ascii=False)

# Perbarui ZIP setelah JSON hasil evaluasi ditambahkan.
if os.path.exists(EXPORT_DIR_LAION + ".zip"):
    os.remove(EXPORT_DIR_LAION + ".zip")
ZIP_PATH = shutil.make_archive(
    EXPORT_DIR_LAION,
    "zip",
    root_dir=EXPORT_DIR_LAION
)

print("Hasil evaluasi LAION berhasil disimpan:")
print(f"JSON : {PATH_OUTPUT_LAION}")
print(f"ZIP  : {ZIP_PATH}")


Hasil evaluasi LAION berhasil disimpan:
JSON : /kaggle/working/LAION_Only_Experiment_Results/laion_experiment_results.json
ZIP  : /kaggle/working/LAION_Only_Experiment_Results.zip


# **Inference**

# M3 — Uji Kasus Sulit: Negasi, Relasi, dan Atribut Budaya

Bagian ini menguji model LAION apakah dapat membedakan query yang mirip secara kosakata tetapi berbeda pada satu aspek utama, seperti negasi, relasi spasial, tindakan, atau atribut budaya.

## Cara menggunakan
1. Jalankan seluruh sel sebelumnya agar dataset, dependensi, dan folder output LAION tersedia.
2. Pada sel konfigurasi, isi `image_id` dari dataset **atau** `image_path` untuk gambar eksternal.
3. Sesuaikan teks kandidat agar benar-benar menggambarkan gambar yang dipilih. Ubah hanya aspek yang sedang diuji.
4. Isi `correct_label` berdasarkan inspeksi gambar **sebelum** menjalankan inferensi. Gunakan `None` bila label benar belum dapat dipastikan.
5. Jalankan sel konfigurasi, fungsi encoder LAION, dan eksekusi M3.

**Interpretasi:** `cosine_similarity` adalah skor kesamaan embedding. `probabilitas_relatif_pct` adalah hasil softmax di antara kandidat pada kasus yang sama, bukan confidence kebenaran yang terkalibrasi.


In [35]:
# [ARSIP NONAKTIF — KONFIGURASI M3 LAMA; diganti oleh revisi M3 LAION dengan query kosong dan laporan softmax.]
# Kode asli dipertahankan di bawah sebagai komentar untuk melacak perubahan.
# # ============================================================
# # KONFIGURASI UJI KASUS SULIT (WAJIB DISESUAIKAN)
# # ============================================================
# 
# # Pilih satu dari dua cara untuk menentukan gambar:
# # 1) image_id: isi dengan nilai image_id dari df_new (gambar dalam dataset validation).
# # 2) image_path: isi dengan path file gambar (boleh di luar dataset); image_id boleh None.
# #
# # Tampilkan sebagian daftar image_id dan nama file dari dataset untuk membantu pemilihan.
# if "df_new" in globals():
#     print("Contoh gambar dalam dataset (salin image_id yang ingin diuji):")
#     display(df_new[["image_id", "file"]].head(20))
# 
# HARD_CASES = [
#     {
#         "case_id": "HC01_NEGASI",
#         "focus": "Negasi: dengan vs tanpa objek",
#         "image_id": "GANTI_IMAGE_ID_1",
#         "image_path": None,
#         "language": "id",  # gunakan "id" atau "en"
#         "candidates": [
#             {"label": "Dengan perkedel", "text": "Sepiring nasi Padang dengan perkedel kentang."},
#             {"label": "Tanpa perkedel", "text": "Sepiring nasi Padang tanpa perkedel kentang."}
#         ],
#         "correct_label": None  # isi label benar setelah memeriksa gambar, sebelum melihat prediksi
#     },
#     {
#         "case_id": "HC02_ATRIBUT_BUDAYA",
#         "focus": "Atribut budaya: warna kostum tari topeng Cirebon",
#         "image_id": "GANTI_IMAGE_ID_2",
#         "image_path": None,
#         "language": "id",
#         "candidates": [
#             {"label": "Kostum merah", "text": "Penari tari topeng Cirebon mengenakan kostum berwarna merah."},
#             {"label": "Kostum biru", "text": "Penari tari topeng Cirebon mengenakan kostum berwarna biru."}
#         ],
#         "correct_label": None
#     },
#     {
#         "case_id": "HC03_RELASI_SPASIAL",
#         "focus": "Relasi spasial: di atas vs di bawah",
#         "image_id": "GANTI_IMAGE_ID_3",
#         "image_path": None,
#         "language": "id",
#         "candidates": [
#             {"label": "Gelas di atas meja", "text": "Sebuah gelas berada di atas meja."},
#             {"label": "Gelas di bawah meja", "text": "Sebuah gelas berada di bawah meja."}
#         ],
#         "correct_label": None
#     },
#     {
#         "case_id": "HC04_RELASI_TINDAKAN",
#         "focus": "Relasi tindakan: memegang vs berada di samping",
#         "image_id": "GANTI_IMAGE_ID_4",
#         "image_path": None,
#         "language": "id",
#         "candidates": [
#             {"label": "Anak memegang payung", "text": "Seorang anak sedang memegang payung di halaman."},
#             {"label": "Payung di samping anak", "text": "Sebuah payung berada di samping seorang anak di halaman."}
#         ],
#         "correct_label": None
#     },
#     {
#         "case_id": "HC05_OBJEK_VS_RELASI",
#         "focus": "Objek tunggal vs relasi manusia-objek",
#         "image_id": "GANTI_IMAGE_ID_5",
#         "image_path": None,
#         "language": "id",
#         "candidates": [
#             {"label": "Hanya sepeda", "text": "Sebuah sepeda berada di taman."},
#             {"label": "Pria mengendarai sepeda", "text": "Seorang pria sedang mengendarai sepeda di taman."}
#         ],
#         "correct_label": None
#     }
# ]
# 
# # [ARSIP NONAKTIF] Daftar model pada eksperimen multi-model sebelumnya:
# # VARIAN_UJI_SULIT = [
# #     "baseline_m1",
# #     "varian1_laion5b",
# #     "varian2_siglip2",
# #     "varian3_nllb_clip"
# # ]
# 
# # [REVISI 2026-11-25] Hanya jalankan LAION.
# VARIAN_UJI_SULIT = ["varian1_laion5b"]
# 
# # Suhu softmax untuk mengubah skor cosine menjadi probabilitas relatif.
# # Gunakan nilai yang sama pada semua model agar prosedur eksperimen konsisten.
# SOFTMAX_TEMPERATURE = 0.07
# 
# # Pemeriksaan konfigurasi dasar.
# assert len(HARD_CASES) == 5, f"Uji ini harus menggunakan tepat 5 gambar, saat ini: {len(HARD_CASES)}"
# assert SOFTMAX_TEMPERATURE > 0, "SOFTMAX_TEMPERATURE harus lebih besar dari 0."
# assert len(VARIAN_UJI_SULIT) > 0, "Pilih minimal satu varian model."
# 
# def resolve_hard_case_image(case):
#     """Membaca gambar dari image_path atau mencari image_id pada df_new."""
#     image_path = case.get("image_path")
#     image_id = case.get("image_id")
# 
#     if image_path:
#         path = os.path.expanduser(str(image_path))
#         if not os.path.isfile(path):
#             raise FileNotFoundError(
#                 f'{case["case_id"]}: file gambar tidak ditemukan: {path}'
#             )
#         with Image.open(path) as im:
#             return im.convert("RGB").copy(), path
# 
#     if image_id is None or str(image_id).startswith("GANTI_"):
#         raise ValueError(
#             f'{case["case_id"]}: isi image_id atau image_path terlebih dahulu.'
#         )
# 
#     if "df_new" not in globals():
#         raise RuntimeError("df_new belum tersedia. Jalankan sel pemuatan dataset terlebih dahulu.")
# 
#     image_id_str = str(image_id)
#     file_names = df_new["file"].astype(str).map(lambda x: os.path.basename(x))
#     mask = (
#         df_new["image_id"].astype(str).eq(image_id_str)
#         | file_names.eq(image_id_str)
#     )
#     matches = df_new.loc[mask]
# 
#     if matches.empty:
#         raise KeyError(
#             f'{case["case_id"]}: image_id/nama file "{image_id_str}" tidak ditemukan di df_new.'
#         )
# 
#     path = str(matches.iloc[0]["file"])
#     if not os.path.isfile(path):
#         raise FileNotFoundError(
#             f'{case["case_id"]}: file pada dataset tidak ditemukan: {path}'
#         )
# 
#     with Image.open(path) as im:
#         return im.convert("RGB").copy(), path
# 
# 
# # Siapkan dan tampilkan lima gambar beserta kandidat query-nya sebelum inferensi.
# prepared_hard_cases = []
# for case in HARD_CASES:
#     if case.get("language") not in ("id", "en"):
#         raise ValueError(f'{case["case_id"]}: language harus "id" atau "en".')
# 
#     candidates = case.get("candidates", [])
#     if len(candidates) < 2:
#         raise ValueError(f'{case["case_id"]}: minimal harus ada dua kandidat query.')
# 
#     labels = [c["label"] for c in candidates]
#     texts = [c["text"] for c in candidates]
#     if len(set(labels)) != len(labels):
#         raise ValueError(f'{case["case_id"]}: label kandidat harus unik.')
#     if any(not str(t).strip() for t in texts):
#         raise ValueError(f'{case["case_id"]}: teks kandidat tidak boleh kosong.')
# 
#     correct_label = case.get("correct_label")
#     if correct_label is not None and correct_label not in labels:
#         raise ValueError(
#             f'{case["case_id"]}: correct_label harus sama dengan salah satu label kandidat.'
#         )
# 
#     image, resolved_path = resolve_hard_case_image(case)
#     prepared_hard_cases.append({
#         **case,
#         "image": image,
#         "resolved_image_path": resolved_path
#     })
# 
#     print(f'\n{case["case_id"]} | Fokus: {case["focus"]} | Bahasa: {case["language"]}')
#     print(f'Gambar: {resolved_path}')
#     display(image.resize((320, 320)))
#     display(pd.DataFrame(candidates))


In [36]:
# [ARSIP NONAKTIF: Loader dan encoder kasus sulit multi-model sebelumnya; diganti oleh encoder LAION-only berikutnya.]

# # ============================================================
# # FUNGSI ENCODER UNTUK INFERENSI GAMBAR vs KANDIDAT QUERY
# # ============================================================
#
# def load_hard_case_encoder(variant_key):
#     """Memuat satu varian model ke CPU. Model dipakai untuk seluruh lima kasus."""
#     if variant_key == "baseline_m1":
#         image_encoder = SentenceTransformer(
#             "sentence-transformers/clip-ViT-B-32",
#             device="cpu"
#         )
#         text_encoder_id = SentenceTransformer(
#             "sentence-transformers/clip-ViT-B-32-multilingual-v1",
#             device="cpu"
#         )
#         image_encoder.eval()
#         text_encoder_id.eval()
#         return {
#             "kind": "sentence_transformers",
#             "image_encoder": image_encoder,
#             "text_encoder_id": text_encoder_id
#         }
#
#     if variant_key == "varian1_laion5b":
#         model, _, preprocess = open_clip.create_model_and_transforms(
#             "xlm-roberta-base-ViT-B-32",
#             pretrained="laion5b_s13b_b90k",
#             device="cpu"
#         )
#         tokenizer = open_clip.get_tokenizer("xlm-roberta-base-ViT-B-32")
#         model.eval()
#         model.requires_grad_(False)
#         return {
#             "kind": "open_clip",
#             "model": model,
#             "preprocess": preprocess,
#             "tokenizer": tokenizer,
#             "language_codes": None
#         }
#
#     if variant_key == "varian2_siglip2":
#         processor = AutoProcessor.from_pretrained("google/siglip2-base-patch16-224")
#         model = AutoModel.from_pretrained("google/siglip2-base-patch16-224")
#         model.eval()
#         model.requires_grad_(False)
#         return {
#             "kind": "siglip2",
#             "model": model,
#             "processor": processor
#         }
#
#     if variant_key == "varian3_nllb_clip":
#         model, _, preprocess = open_clip.create_model_and_transforms(
#             "nllb-clip-base-siglip",
#             pretrained="v1",
#             device="cpu"
#         )
#         tokenizer = open_clip.get_tokenizer("nllb-clip-base-siglip")
#         model.eval()
#         model.requires_grad_(False)
#         return {
#             "kind": "open_clip",
#             "model": model,
#             "preprocess": preprocess,
#             "tokenizer": tokenizer,
#             "language_codes": {"id": "ind_Latn", "en": "eng_Latn"}
#         }
#
#     raise ValueError(f"Varian tidak dikenal: {variant_key}")
#
#
# def _hf_features_to_numpy(outputs):
#     """Mengambil pooled features dari output Transformers dan menormalkannya."""
#     if hasattr(outputs, "pooler_output") and outputs.pooler_output is not None:
#         features = outputs.pooler_output
#     elif hasattr(outputs, "image_embeds") and outputs.image_embeds is not None:
#         features = outputs.image_embeds
#     elif torch.is_tensor(outputs):
#         features = outputs
#     else:
#         raise TypeError(
#             "Output encoder Transformers tidak memiliki pooler_output/image_embeds."
#         )
#
#     features = F.normalize(features.float(), p=2, dim=-1)
#     return features.detach().cpu().numpy()
#
#
# def encode_hard_case(encoder, image, texts, language):
#     """Menghasilkan satu embedding gambar dan embedding untuk seluruh kandidat query."""
#     if encoder["kind"] == "sentence_transformers":
#         text_encoder = (
#             encoder["text_encoder_id"]
#             if language == "id"
#             else encoder["image_encoder"]
#         )
#
#         image_embedding = encoder["image_encoder"].encode(
#             [image],
#             convert_to_numpy=True,
#             normalize_embeddings=True,
#             show_progress_bar=False
#         )
#         text_embeddings = text_encoder.encode(
#             texts,
#             convert_to_numpy=True,
#             normalize_embeddings=True,
#             show_progress_bar=False
#         )
#
#     elif encoder["kind"] == "open_clip":
#         model = encoder["model"]
#         tokenizer = encoder["tokenizer"]
#
#         language_codes = encoder.get("language_codes")
#         if language_codes:
#             tokenizer.set_language(language_codes[language])
#
#         image_tensor = encoder["preprocess"](image).unsqueeze(0)
#         text_tokens = tokenizer(texts)
#
#         with torch.no_grad():
#             image_features = model.encode_image(image_tensor)
#             text_features = model.encode_text(text_tokens)
#
#             image_embedding = F.normalize(
#                 image_features.float(), p=2, dim=-1
#             ).cpu().numpy()
#             text_embeddings = F.normalize(
#                 text_features.float(), p=2, dim=-1
#             ).cpu().numpy()
#
#     elif encoder["kind"] == "siglip2":
#         processor = encoder["processor"]
#         model = encoder["model"]
#
#         image_inputs = processor(images=[image], return_tensors="pt")
#         text_inputs = processor(
#             text=texts,
#             padding="max_length",
#             truncation=True,
#             return_tensors="pt"
#         )
#
#         with torch.no_grad():
#             image_outputs = model.get_image_features(**image_inputs)
#             text_outputs = model.get_text_features(**text_inputs)
#
#             image_embedding = _hf_features_to_numpy(image_outputs)
#             text_embeddings = _hf_features_to_numpy(text_outputs)
#
#     else:
#         raise ValueError(f"Jenis encoder tidak didukung: {encoder['kind']}")
#
#     image_embedding = np.asarray(image_embedding, dtype=np.float32)
#     text_embeddings = np.asarray(text_embeddings, dtype=np.float32)
#
#     if image_embedding.ndim != 2 or image_embedding.shape[0] != 1:
#         raise ValueError(f"Shape embedding gambar tidak sesuai: {image_embedding.shape}")
#     if text_embeddings.ndim != 2 or text_embeddings.shape[0] != len(texts):
#         raise ValueError(f"Shape embedding teks tidak sesuai: {text_embeddings.shape}")
#     if image_embedding.shape[1] != text_embeddings.shape[1]:
#         raise ValueError(
#             "Dimensi embedding gambar dan teks berbeda: "
#             f"{image_embedding.shape[1]} vs {text_embeddings.shape[1]}"
#         )
#     if not np.isfinite(image_embedding).all() or not np.isfinite(text_embeddings).all():
#         raise ValueError("Embedding mengandung NaN atau Inf.")
#
#     return image_embedding, text_embeddings
#
#
# def softmax_relative_probability(similarities, temperature=0.07):
#     """
#     Softmax atas cosine similarity untuk satu himpunan kandidat.
#     Nilainya adalah probabilitas relatif antar kandidat, bukan confidence terkalibrasi.
#     """
#     scores = np.asarray(similarities, dtype=np.float64) / float(temperature)
#     scores = scores - np.max(scores)
#     exp_scores = np.exp(scores)
#     return exp_scores / exp_scores.sum()


### LAION hard-case encoder baru

In [37]:
# ============================================================
# [REVISI 2026-11-25] FUNGSI ENCODER UJI KASUS SULIT — LAION SAJA
# ============================================================

def load_hard_case_encoder(variant_key=LAION_VARIANT_KEY):
    """Memuat encoder LAION satu kali untuk seluruh kasus sulit."""
    if variant_key != LAION_VARIANT_KEY:
        raise ValueError(
            f"Revisi ini hanya mendukung {LAION_VARIANT_KEY}, bukan {variant_key}."
        )

    model, _, preprocess = open_clip.create_model_and_transforms(
        LAION_ARCH_NAME,
        pretrained=LAION_PRETRAINED_NAME,
        device="cpu"
    )
    tokenizer = open_clip.get_tokenizer(LAION_ARCH_NAME)
    model.eval()
    model.requires_grad_(False)

    return {
        "kind": "open_clip_laion",
        "model": model,
        "preprocess": preprocess,
        "tokenizer": tokenizer
    }


def encode_hard_case(encoder, image, texts, language):
    """Menghasilkan embedding gambar dan embedding kandidat query memakai LAION."""
    if encoder.get("kind") != "open_clip_laion":
        raise ValueError("Encoder yang diberikan bukan encoder LAION.")
    if language not in ("id", "en"):
        raise ValueError("language harus bernilai 'id' atau 'en'.")
    if len(texts) < 2:
        raise ValueError("Setiap kasus harus memiliki minimal dua kandidat query.")

    image_tensor = encoder["preprocess"](image).unsqueeze(0)
    text_tokens = encoder["tokenizer"]([str(text) for text in texts])

    with torch.no_grad():
        image_features = encoder["model"].encode_image(image_tensor)
        text_features = encoder["model"].encode_text(text_tokens)

        image_embedding = F.normalize(image_features.float(), p=2, dim=-1).cpu().numpy()
        text_embeddings = F.normalize(text_features.float(), p=2, dim=-1).cpu().numpy()

    image_embedding = np.asarray(image_embedding, dtype=np.float32)
    text_embeddings = np.asarray(text_embeddings, dtype=np.float32)

    if image_embedding.shape != (1, text_embeddings.shape[1]):
        raise ValueError(
            f"Shape embedding gambar tidak sesuai: {image_embedding.shape}"
        )
    if text_embeddings.ndim != 2 or text_embeddings.shape[0] != len(texts):
        raise ValueError(
            f"Shape embedding kandidat tidak sesuai: {text_embeddings.shape}"
        )
    if not np.isfinite(image_embedding).all() or not np.isfinite(text_embeddings).all():
        raise ValueError("Embedding mengandung NaN atau Inf.")

    return image_embedding, text_embeddings


def softmax_relative_probability(similarities, temperature=0.07):
    """Softmax skor kandidat; nilai adalah probabilitas relatif, bukan confidence terkalibrasi."""
    if temperature <= 0:
        raise ValueError("temperature harus lebih besar dari 0.")
    scores = np.asarray(similarities, dtype=np.float64) / float(temperature)
    scores = scores - np.max(scores)
    exp_scores = np.exp(scores)
    return exp_scores / exp_scores.sum()

print("Fungsi inferensi kasus sulit LAION siap.")


Fungsi inferensi kasus sulit LAION siap.


In [38]:
# [ARSIP NONAKTIF: Runner kasus sulit multi-model sebelumnya; diganti oleh runner LAION-only berikutnya.]

# # ============================================================
# # JALANKAN UJI KASUS SULIT DAN SIMPAN HASIL
# # ============================================================
#
# hard_case_rows = []
# hard_case_summary_rows = []
#
# for variant_key in VARIAN_UJI_SULIT:
#     print(f"\n{'=' * 78}\nMenjalankan uji kasus sulit: {variant_key}\n{'=' * 78}")
#     encoder = load_hard_case_encoder(variant_key)
#
#     try:
#         for case in prepared_hard_cases:
#             candidate_texts = [c["text"] for c in case["candidates"]]
#             candidate_labels = [c["label"] for c in case["candidates"]]
#
#             image_embedding, text_embeddings = encode_hard_case(
#                 encoder=encoder,
#                 image=case["image"],
#                 texts=candidate_texts,
#                 language=case["language"]
#             )
#
#             # Kedua sisi sudah L2-normalized, sehingga dot product = cosine similarity.
#             similarities = (image_embedding @ text_embeddings.T).reshape(-1)
#             probabilities = softmax_relative_probability(
#                 similarities,
#                 temperature=SOFTMAX_TEMPERATURE
#             )
#
#             ranking = np.argsort(-probabilities)
#             top_idx = int(ranking[0])
#             second_idx = int(ranking[1])
#
#             correct_label = case.get("correct_label")
#             top1_correct = (
#                 candidate_labels[top_idx] == correct_label
#                 if correct_label is not None
#                 else None
#             )
#
#             hard_case_summary_rows.append({
#                 "varian": variant_key,
#                 "case_id": case["case_id"],
#                 "fokus_uji": case["focus"],
#                 "bahasa_query": case["language"],
#                 "path_gambar": case["resolved_image_path"],
#                 "kandidat_teratas": candidate_labels[top_idx],
#                 "probabilitas_teratas_pct": round(float(probabilities[top_idx] * 100), 2),
#                 "selisih_top1_top2_pct": round(
#                     float((probabilities[top_idx] - probabilities[second_idx]) * 100), 2
#                 ),
#                 "correct_label": correct_label,
#                 "top1_benar": top1_correct
#             })
#
#             for rank_pos, candidate_idx in enumerate(ranking, start=1):
#                 candidate = case["candidates"][int(candidate_idx)]
#                 hard_case_rows.append({
#                     "varian": variant_key,
#                     "case_id": case["case_id"],
#                     "fokus_uji": case["focus"],
#                     "bahasa_query": case["language"],
#                     "path_gambar": case["resolved_image_path"],
#                     "label_kandidat": candidate["label"],
#                     "query": candidate["text"],
#                     "cosine_similarity": round(float(similarities[candidate_idx]), 6),
#                     "probabilitas_relatif_pct": round(
#                         float(probabilities[candidate_idx] * 100), 2
#                     ),
#                     "peringkat": rank_pos,
#                     "correct_label": correct_label,
#                     "kandidat_benar": (
#                         candidate["label"] == correct_label
#                         if correct_label is not None
#                         else None
#                     )
#                 })
#
#             print(
#                 f'{case["case_id"]}: kandidat teratas = {candidate_labels[top_idx]} '
#                 f'({probabilities[top_idx] * 100:.2f}%)'
#             )
#
#     finally:
#         # Lepaskan model sebelum memuat varian berikutnya agar RAM tetap terkendali.
#         del encoder
#         gc.collect()
#         if torch.cuda.is_available():
#             torch.cuda.empty_cache()
#
# df_hard_case_probabilities = pd.DataFrame(hard_case_rows)
# df_hard_case_summary = pd.DataFrame(hard_case_summary_rows)
#
# print("\n=== PROBABILITAS SETIAP KANDIDAT ===")
# display(
#     df_hard_case_probabilities.sort_values(
#         ["varian", "case_id", "peringkat"]
#     ).reset_index(drop=True)
# )
#
# print("\n=== RINGKASAN SETIAP KASUS ===")
# display(
#     df_hard_case_summary.sort_values(
#         ["varian", "case_id"]
#     ).reset_index(drop=True)
# )
#
# # Akurasi Top-1 hanya dihitung untuk kasus yang correct_label-nya sudah diisi.
# annotated = df_hard_case_summary.dropna(subset=["top1_benar"])
# if not annotated.empty:
#     top1_by_variant = (
#         annotated.groupby("varian")["top1_benar"]
#         .mean()
#         .mul(100)
#         .round(2)
#         .rename("top1_accuracy_pct")
#         .reset_index()
#     )
#     print("\n=== TOP-1 PADA KASUS YANG SUDAH DIBERI LABEL BENAR ===")
#     display(top1_by_variant)
# else:
#     print(
#         "\nTop-1 accuracy belum dihitung karena seluruh correct_label masih None. "
#         "Isi label benar sebelum inferensi dan jalankan kembali untuk memperoleh metrik ini."
#     )
#
# # Simpan tabel agar dapat dipakai dalam laporan eksperimen.
# HARD_CASE_EXPORT_DIR = EXPORT_DIR if "EXPORT_DIR" in globals() else "/kaggle/working/M2_Experiment_Results"
# os.makedirs(HARD_CASE_EXPORT_DIR, exist_ok=True)
#
# hard_prob_path = os.path.join(
#     HARD_CASE_EXPORT_DIR, "hard_case_probabilities.csv"
# )
# hard_summary_path = os.path.join(
#     HARD_CASE_EXPORT_DIR, "hard_case_summary.csv"
# )
#
# df_hard_case_probabilities.to_csv(hard_prob_path, index=False, encoding="utf-8-sig")
# df_hard_case_summary.to_csv(hard_summary_path, index=False, encoding="utf-8-sig")
#
# print("\nHasil uji kasus sulit berhasil disimpan:")
# print(f"- {hard_prob_path}")
# print(f"- {hard_summary_path}")
#
# # Perbarui ZIP hasil eksperimen agar dua CSV uji kasus sulit ikut terarsip.
# if "shutil" not in globals():
#     import shutil
#
# zip_path = HARD_CASE_EXPORT_DIR.rstrip("/\\") + ".zip"
# if os.path.exists(zip_path):
#     os.remove(zip_path)
# ZIP_PATH = shutil.make_archive(
#     HARD_CASE_EXPORT_DIR.rstrip("/\\"),
#     "zip",
#     root_dir=HARD_CASE_EXPORT_DIR
# )
# print(f"- Arsip terbaru: {ZIP_PATH}")


### LAION hard-case runner baru

In [39]:
# [ARSIP NONAKTIF — RUNNER M3 LAMA; diganti oleh revisi M3 LAION dengan query kosong dan laporan softmax.]
# Kode asli dipertahankan di bawah sebagai komentar untuk melacak perubahan.
# # ============================================================
# # [REVISI 2026-11-25] JALANKAN UJI KASUS SULIT — LAION SAJA
# # ============================================================
# 
# assert VARIAN_UJI_SULIT == [LAION_VARIANT_KEY], (
#     "Konfigurasi revisi ini harus menjalankan LAION saja."
# )
# 
# hard_case_rows = []
# hard_case_summary_rows = []
# 
# print(f"\n{'=' * 78}\nMenjalankan uji kasus sulit: {LAION_VARIANT_KEY}\n{'=' * 78}")
# encoder = load_hard_case_encoder(LAION_VARIANT_KEY)
# 
# try:
#     for case in prepared_hard_cases:
#         candidate_texts = [candidate["text"] for candidate in case["candidates"]]
#         candidate_labels = [candidate["label"] for candidate in case["candidates"]]
# 
#         image_embedding, text_embeddings = encode_hard_case(
#             encoder=encoder,
#             image=case["image"],
#             texts=candidate_texts,
#             language=case["language"]
#         )
# 
#         # Kedua sisi telah L2-normalized, sehingga dot product sama dengan cosine similarity.
#         similarities = (image_embedding @ text_embeddings.T).reshape(-1)
#         probabilities = softmax_relative_probability(
#             similarities,
#             temperature=SOFTMAX_TEMPERATURE
#         )
# 
#         ranking = np.argsort(-probabilities)
#         top_idx = int(ranking[0])
#         second_idx = int(ranking[1])
#         correct_label = case.get("correct_label")
#         top1_correct = (
#             candidate_labels[top_idx] == correct_label
#             if correct_label is not None
#             else None
#         )
# 
#         hard_case_summary_rows.append({
#             "varian": LAION_VARIANT_KEY,
#             "checkpoint": LAION_CHECKPOINT,
#             "case_id": case["case_id"],
#             "fokus_uji": case["focus"],
#             "bahasa_query": case["language"],
#             "path_gambar": case["resolved_image_path"],
#             "kandidat_teratas": candidate_labels[top_idx],
#             "probabilitas_teratas_pct": round(float(probabilities[top_idx] * 100), 2),
#             "selisih_top1_top2_pct": round(
#                 float((probabilities[top_idx] - probabilities[second_idx]) * 100), 2
#             ),
#             "correct_label": correct_label,
#             "top1_benar": top1_correct
#         })
# 
#         for rank_pos, candidate_idx in enumerate(ranking, start=1):
#             candidate = case["candidates"][int(candidate_idx)]
#             hard_case_rows.append({
#                 "varian": LAION_VARIANT_KEY,
#                 "checkpoint": LAION_CHECKPOINT,
#                 "case_id": case["case_id"],
#                 "fokus_uji": case["focus"],
#                 "bahasa_query": case["language"],
#                 "path_gambar": case["resolved_image_path"],
#                 "label_kandidat": candidate["label"],
#                 "query": candidate["text"],
#                 "cosine_similarity": round(float(similarities[candidate_idx]), 6),
#                 "probabilitas_relatif_pct": round(
#                     float(probabilities[candidate_idx] * 100), 2
#                 ),
#                 "peringkat": rank_pos,
#                 "correct_label": correct_label,
#                 "kandidat_benar": (
#                     candidate["label"] == correct_label
#                     if correct_label is not None
#                     else None
#                 )
#             })
# 
#         print(
#             f'{case["case_id"]}: kandidat teratas = {candidate_labels[top_idx]} '
#             f'({probabilities[top_idx] * 100:.2f}%)'
#         )
# finally:
#     del encoder
#     gc.collect()
#     if torch.cuda.is_available():
#         torch.cuda.empty_cache()
# 
# 
# df_hard_case_probabilities = pd.DataFrame(hard_case_rows)
# df_hard_case_summary = pd.DataFrame(hard_case_summary_rows)
# 
# print("\n=== PROBABILITAS SETIAP KANDIDAT — LAION ===")
# display(
#     df_hard_case_probabilities.sort_values(
#         ["case_id", "peringkat"]
#     ).reset_index(drop=True)
# )
# 
# print("\n=== RINGKASAN SETIAP KASUS — LAION ===")
# display(
#     df_hard_case_summary.sort_values("case_id").reset_index(drop=True)
# )
# 
# annotated = df_hard_case_summary.dropna(subset=["top1_benar"])
# if not annotated.empty:
#     top1_accuracy = annotated["top1_benar"].astype(bool).mean() * 100
#     print(f"Top-1 accuracy pada kasus berlabel: {top1_accuracy:.2f}%")
# else:
#     print(
#         "Top-1 accuracy belum dihitung karena correct_label belum diisi. "
#         "Isi label benar sebelum inferensi untuk evaluasi akurasi."
#     )
# 
# HARD_CASE_EXPORT_DIR = EXPORT_DIR_LAION
# hard_prob_path = os.path.join(
#     HARD_CASE_EXPORT_DIR, "hard_case_probabilities.csv"
# )
# hard_summary_path = os.path.join(
#     HARD_CASE_EXPORT_DIR, "hard_case_summary.csv"
# )
# df_hard_case_probabilities.to_csv(hard_prob_path, index=False, encoding="utf-8-sig")
# df_hard_case_summary.to_csv(hard_summary_path, index=False, encoding="utf-8-sig")
# 
# # Perbarui ZIP agar seluruh hasil LAION dan hasil M3 ikut terarsip.
# if os.path.exists(EXPORT_DIR_LAION + ".zip"):
#     os.remove(EXPORT_DIR_LAION + ".zip")
# ZIP_PATH = shutil.make_archive(
#     EXPORT_DIR_LAION,
#     "zip",
#     root_dir=EXPORT_DIR_LAION
# )
# 
# print("\nHasil uji kasus sulit LAION disimpan:")
# print(f"- {hard_prob_path}")
# print(f"- {hard_summary_path}")
# print(f"- {ZIP_PATH}")


# M3 — Revisi pelaporan hasil uji kasus sulit (LAION-only)

Bagian ini menerapkan format pelaporan instruksi M3. Untuk setiap gambar, dua kandidat query dibandingkan menggunakan cosine similarity. Skor untuk **dua kandidat pada gambar yang sama** dinormalisasi dengan softmax standar:

\[P_i = \frac{e^{s_i}}{\sum_j e^{s_j}}\]

Nilai tersebut adalah preferensi relatif antara kandidat, **bukan probabilitas kebenaran yang terkalibrasi**. Kasus yang diprediksi salah tetap masuk ke tabel dan file ekspor.

**Template query sengaja dikosongkan.** Isi `image_id` atau `image_path`, kemudian isi teks pada `Kandidat benar` dan `Pengecoh`. Pastikan kandidat benar sudah ditentukan dari inspeksi gambar sebelum inferensi. Sel yang belum lengkap akan dilewati tanpa menghentikan notebook.


**Catatan pemilihan model:** instruksi M3 meminta model terbaik berdasarkan hasil M2. Karena notebook ini telah dibatasi ke LAION-only, sel M3 menggunakan LAION sebagai model terpilih. Pastikan hasil M2 memang menunjukkan LAION sebagai sistem terbaik sebelum menyimpulkan bahwa persyaratan tersebut terpenuhi; notebook ini tidak lagi membandingkan LAION dengan model lain.


In [40]:
# ============================================================
# [REVISI 2026-11-25] TEMPLATE KASUS SULIT M3 — LAION SAJA
# Query dan pemilihan gambar sengaja dikosongkan terlebih dahulu.
# ============================================================

M3_HARD_CASES = [
    {
        "case_id": "HC01_NEGASI",
        "focus": "Negasi: dengan vs tanpa objek",
        "image_id": "",          # isi image_id dari df_new, atau biarkan kosong bila memakai image_path
        "image_path": None,        # isi path gambar eksternal bila tidak memakai image_id
        "language": "id",         # "id" atau "en"
        "candidates": [
            {"label": "Kandidat benar", "text": ""},
            {"label": "Pengecoh", "text": ""}
        ],
        # Pastikan teks pada kandidat ini benar-benar sesuai gambar sebelum menjalankan model.
        "correct_label": "Kandidat benar"
    },
    {
        "case_id": "HC02_ATRIBUT_BUDAYA",
        "focus": "Atribut budaya",
        "image_id": "",
        "image_path": None,
        "language": "id",
        "candidates": [
            {"label": "Kandidat benar", "text": ""},
            {"label": "Pengecoh", "text": ""}
        ],
        "correct_label": "Kandidat benar"
    },
    {
        "case_id": "HC03_RELASI_SPASIAL",
        "focus": "Relasi spasial",
        "image_id": "",
        "image_path": None,
        "language": "id",
        "candidates": [
            {"label": "Kandidat benar", "text": ""},
            {"label": "Pengecoh", "text": ""}
        ],
        "correct_label": "Kandidat benar"
    },
    {
        "case_id": "HC04_RELASI_TINDAKAN",
        "focus": "Relasi tindakan",
        "image_id": "",
        "image_path": None,
        "language": "id",
        "candidates": [
            {"label": "Kandidat benar", "text": ""},
            {"label": "Pengecoh", "text": ""}
        ],
        "correct_label": "Kandidat benar"
    },
    {
        "case_id": "HC05_OBJEK_VS_RELASI",
        "focus": "Objek vs relasi",
        "image_id": "",
        "image_path": None,
        "language": "id",
        "candidates": [
            {"label": "Kandidat benar", "text": ""},
            {"label": "Pengecoh", "text": ""}
        ],
        "correct_label": "Kandidat benar"
    }
]

# Notebook ini hanya menjalankan LAION, sesuai revisi eksperimen sebelumnya.
M3_MODEL_VARIANT = LAION_VARIANT_KEY
M3_SOFTMAX_TEMPERATURE = 1.0  # softmax standar sesuai rumus pada instruksi M3
assert M3_MODEL_VARIANT == "varian1_laion5b"
assert len(M3_HARD_CASES) == 5, "Template M3 harus berisi tepat lima kasus."


def m3_resolve_image(case):
    """Ambil gambar dari image_path atau image_id pada dataframe df_new."""
    image_path = case.get("image_path")
    image_id = case.get("image_id")

    if image_path:
        path = os.path.expanduser(str(image_path))
        if not os.path.isfile(path):
            raise FileNotFoundError(f'{case["case_id"]}: gambar tidak ditemukan: {path}')
        with Image.open(path) as image:
            return image.convert("RGB").copy(), path

    if image_id is None or not str(image_id).strip():
        raise ValueError("image_id atau image_path belum diisi.")
    if "df_new" not in globals():
        raise RuntimeError("df_new belum tersedia. Jalankan sel pemuatan dataset terlebih dahulu.")

    image_id = str(image_id).strip()
    required_columns = {"image_id", "file"}
    if not required_columns.issubset(df_new.columns):
        raise KeyError("df_new harus memiliki kolom image_id dan file.")

    file_names = df_new["file"].astype(str).map(os.path.basename)
    matches = df_new.loc[
        df_new["image_id"].astype(str).eq(image_id) | file_names.eq(image_id)
    ]
    if matches.empty:
        raise KeyError(f'{case["case_id"]}: image_id/nama file "{image_id}" tidak ditemukan.')

    path = str(matches.iloc[0]["file"])
    if not os.path.isfile(path):
        raise FileNotFoundError(f'{case["case_id"]}: file dataset tidak ditemukan: {path}')
    with Image.open(path) as image:
        return image.convert("RGB").copy(), path


# Validasi ringan dan pemisahan kasus siap vs belum siap.
M3_PREPARED_CASES = []
M3_SKIPPED_CASES = []

for case in M3_HARD_CASES:
    candidates = case.get("candidates", [])
    if len(candidates) != 2:
        M3_SKIPPED_CASES.append({
            "case_id": case.get("case_id", "TANPA_ID"),
            "alasan": "Harus tersedia tepat dua kandidat: kandidat benar dan pengecoh."
        })
        continue

    if case.get("language") not in ("id", "en"):
        M3_SKIPPED_CASES.append({
            "case_id": case["case_id"],
            "alasan": 'language harus "id" atau "en".'
        })
        continue

    labels = [candidate.get("label") for candidate in candidates]
    texts = [str(candidate.get("text", "")).strip() for candidate in candidates]
    correct_label = case.get("correct_label")

    if len(set(labels)) != 2 or correct_label not in labels:
        M3_SKIPPED_CASES.append({
            "case_id": case["case_id"],
            "alasan": "Label kandidat harus unik dan correct_label harus cocok dengan salah satunya."
        })
        continue

    missing = []
    if not all(texts):
        missing.append("teks kandidat belum lengkap")
    if not (case.get("image_path") or str(case.get("image_id") or "").strip()):
        missing.append("image_id/image_path belum diisi")

    if missing:
        M3_SKIPPED_CASES.append({
            "case_id": case["case_id"],
            "alasan": "; ".join(missing)
        })
        continue

    # Simpan teks yang sudah dibersihkan tanpa mengubah konfigurasi aslinya.
    ready_case = {**case, "candidates": [
        {**candidate, "text": text}
        for candidate, text in zip(candidates, texts)
    ]}
    try:
        image, resolved_path = m3_resolve_image(ready_case)
    except Exception as exc:
        M3_SKIPPED_CASES.append({
            "case_id": case["case_id"],
            "alasan": f"Gambar belum dapat dimuat: {exc}"
        })
        continue

    ready_case["image"] = image
    ready_case["resolved_image_path"] = resolved_path
    M3_PREPARED_CASES.append(ready_case)

print(f"Kasus siap dijalankan: {len(M3_PREPARED_CASES)} dari {len(M3_HARD_CASES)}")
if M3_SKIPPED_CASES:
    print("Kasus yang dilewati / masih perlu dilengkapi:")
    display(pd.DataFrame(M3_SKIPPED_CASES))

if M3_PREPARED_CASES:
    print("\nPreview gambar dan kandidat untuk kasus yang siap:")
    for case in M3_PREPARED_CASES:
        print(f'\n{case["case_id"]} | {case["focus"]} | {case["resolved_image_path"]}')
        display(case["image"].resize((320, 320)))
        display(pd.DataFrame(case["candidates"]))
else:
    print(
        "Belum ada inferensi M3 yang dijalankan. Isi image_id/image_path dan kedua teks query "
        "pada minimal satu kasus, lalu jalankan ulang sel konfigurasi dan sel runner di bawah."
    )


Kasus siap dijalankan: 0 dari 5
Kasus yang dilewati / masih perlu dilengkapi:


,case_id,alasan
0,HC01_NEGASI,teks kandidat belum lengkap; image_id/image_pa...
1,HC02_ATRIBUT_BUDAYA,teks kandidat belum lengkap; image_id/image_pa...
2,HC03_RELASI_SPASIAL,teks kandidat belum lengkap; image_id/image_pa...
3,HC04_RELASI_TINDAKAN,teks kandidat belum lengkap; image_id/image_pa...
4,HC05_OBJEK_VS_RELASI,teks kandidat belum lengkap; image_id/image_pa...


Belum ada inferensi M3 yang dijalankan. Isi image_id/image_path dan kedua teks query pada minimal satu kasus, lalu jalankan ulang sel konfigurasi dan sel runner di bawah.


### Cara mengisi template M3

- Isi `image_id` dari dataframe validation atau `image_path` untuk gambar eksternal.
- Isi `text` pada `Kandidat benar` dengan query yang sesuai gambar.
- Isi `text` pada `Pengecoh` dengan query pembanding yang salah pada aspek yang diuji.
- Jangan menukar peran label tanpa memperbarui `correct_label`. Tetapkan kandidat benar melalui inspeksi gambar sebelum melihat hasil model.
- Jalankan ulang sel konfigurasi M3, lalu sel inferensi dan laporan. Kasus yang belum lengkap akan dilewati; kasus yang lengkap akan diuji.
- Rumus softmax menggunakan skor dua kandidat untuk gambar tersebut saja. `Lolos?` bernilai `Tidak lolos` saat probabilitas pengecoh lebih tinggi; kegagalan tetap disimpan.


In [41]:
# ============================================================
# [REVISI 2026-11-25] INFERENSI, SOFTMAX, DAN LAPORAN M3 — LAION
# ============================================================

# Fungsi softmax standar: tidak memakai temperature tambahan, sesuai rumus instruksi.
def m3_standard_softmax(scores):
    scores = np.asarray(scores, dtype=np.float64).reshape(-1)
    if scores.size < 2:
        raise ValueError("Softmax M3 membutuhkan minimal dua kandidat.")
    if not np.isfinite(scores).all():
        raise ValueError("Skor similarity mengandung NaN atau Inf.")
    exp_scores = np.exp(scores - np.max(scores))
    probabilities = exp_scores / exp_scores.sum()
    return probabilities


m3_report_rows = []
m3_detail_rows = []

if not M3_PREPARED_CASES:
    # Tampilkan struktur laporan tanpa menjalankan atau memuat model.
    df_m3_report = pd.DataFrame(columns=[
        "Query", "P(kandidat benar)", "P(pengecoh)", "Lolos?"
    ])
    df_m3_details = pd.DataFrame(columns=[
        "case_id", "peran_kandidat", "Query", "cosine_similarity",
        "softmax_probability_pct", "peringkat", "dipilih_model", "kandidat_benar"
    ])
    print("Tabel laporan M3 masih kosong karena belum ada kasus yang siap diuji.")
    display(df_m3_report)
else:
    assert M3_MODEL_VARIANT == LAION_VARIANT_KEY, "M3 wajib memakai model LAION pada notebook ini."
    print(f"Menjalankan {len(M3_PREPARED_CASES)} kasus menggunakan {LAION_CHECKPOINT}")
    m3_encoder = load_hard_case_encoder(M3_MODEL_VARIANT)

    try:
        for case in M3_PREPARED_CASES:
            texts = [candidate["text"] for candidate in case["candidates"]]
            labels = [candidate["label"] for candidate in case["candidates"]]
            correct_label = case["correct_label"]

            image_embedding, text_embeddings = encode_hard_case(
                encoder=m3_encoder,
                image=case["image"],
                texts=texts,
                language=case["language"]
            )

            # Embedding sudah dinormalisasi; dot product = cosine similarity.
            similarities = (image_embedding @ text_embeddings.T).reshape(-1)
            probabilities = m3_standard_softmax(similarities)

            correct_idx = labels.index(correct_label)
            distractor_idx = 1 - correct_idx  # tepat dua kandidat untuk setiap gambar
            predicted_idx = int(np.argmax(probabilities))
            passed = predicted_idx == correct_idx
            status = "Lolos" if passed else "Tidak lolos"

            correct_text = texts[correct_idx]
            distractor_text = texts[distractor_idx]
            query_pair = (
                f"Kandidat benar: {correct_text}\n"
                f"Pengecoh: {distractor_text}"
            )

            # Format utama sesuai kolom pada template laporan instruksi.
            m3_report_rows.append({
                "case_id": case["case_id"],
                "Query": query_pair,
                "P(kandidat benar)": round(float(probabilities[correct_idx] * 100), 2),
                "P(pengecoh)": round(float(probabilities[distractor_idx] * 100), 2),
                "Lolos?": status,
                "Prediksi model": labels[predicted_idx],
                "path_gambar": case["resolved_image_path"]
            })

            # Tabel detail menyimpan semua kandidat, termasuk ketika model memilih pengecoh.
            for candidate_idx, candidate in enumerate(case["candidates"]):
                m3_detail_rows.append({
                    "case_id": case["case_id"],
                    "peran_kandidat": candidate["label"],
                    "Query": candidate["text"],
                    "cosine_similarity": round(float(similarities[candidate_idx]), 6),
                    "softmax_probability_pct": round(float(probabilities[candidate_idx] * 100), 2),
                    "peringkat": int(np.argsort(-probabilities).tolist().index(candidate_idx) + 1),
                    "dipilih_model": candidate_idx == predicted_idx,
                    "kandidat_benar": candidate_idx == correct_idx
                })

            print(
                f'{case["case_id"]}: {status} | '
                f'P(benar)={probabilities[correct_idx] * 100:.2f}% | '
                f'P(pengecoh)={probabilities[distractor_idx] * 100:.2f}% | '
                f'prediksi={labels[predicted_idx]}'
            )
    finally:
        del m3_encoder
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    df_m3_report = pd.DataFrame(m3_report_rows)
    df_m3_details = pd.DataFrame(m3_detail_rows)

    print("\n=== TABEL LAPORAN M3 — FORMAT INSTRUKSI ===")
    display(df_m3_report[[
        "Query", "P(kandidat benar)", "P(pengecoh)", "Lolos?"
    ]])

    print("\n=== DETAIL SELURUH KANDIDAT (KEBERHASILAN DAN KEGAGALAN) ===")
    display(df_m3_details.sort_values(["case_id", "peringkat"]).reset_index(drop=True))

    total_cases = len(df_m3_report)
    passed_cases = int((df_m3_report["Lolos?"] == "Lolos").sum())
    failed_cases = int((df_m3_report["Lolos?"] == "Tidak lolos").sum())
    print(
        f"\nRingkasan M3: {passed_cases}/{total_cases} lolos; "
        f"{failed_cases}/{total_cases} tidak lolos. Semua kasus tetap ditampilkan."
    )

    # Simpan hasil CSV dan JSON ke folder khusus LAION.
    M3_EXPORT_DIR = globals().get(
        "EXPORT_DIR_LAION",
        "/kaggle/working/LAION_Only_Experiment_Results"
    )
    os.makedirs(M3_EXPORT_DIR, exist_ok=True)

    m3_report_path = os.path.join(M3_EXPORT_DIR, "m3_hard_case_report.csv")
    m3_detail_path = os.path.join(M3_EXPORT_DIR, "m3_hard_case_candidate_details.csv")
    m3_json_path = os.path.join(M3_EXPORT_DIR, "m3_hard_case_report.json")

    df_m3_report.to_csv(m3_report_path, index=False, encoding="utf-8-sig")
    df_m3_details.to_csv(m3_detail_path, index=False, encoding="utf-8-sig")

    m3_export_payload = {
        "model": {
            "variant": LAION_VARIANT_KEY,
            "checkpoint": LAION_CHECKPOINT
        },
        "softmax": {
            "method": "standard_softmax_per_image",
            "temperature": 1.0,
            "formula": "exp(similarity_i) / sum(exp(similarity_j))",
            "interpretation": "relative preference among candidates for the same image; not calibrated confidence"
        },
        "summary": {
            "total_cases": total_cases,
            "passed_cases": passed_cases,
            "failed_cases": failed_cases,
            "top1_accuracy_pct": round(passed_cases / total_cases * 100, 2) if total_cases else None
        },
        "results": m3_report_rows,
        "candidate_details": m3_detail_rows,
        "skipped_cases": M3_SKIPPED_CASES
    }
    with open(m3_json_path, "w", encoding="utf-8") as f:
        json.dump(m3_export_payload, f, indent=4, ensure_ascii=False)

    # Perbarui ZIP jika modul shutil tersedia (sudah dipakai pada sel ekspor M2).
    if "shutil" in globals():
        m3_zip_path = M3_EXPORT_DIR.rstrip("/\\") + ".zip"
        if os.path.exists(m3_zip_path):
            os.remove(m3_zip_path)
        m3_zip_path = shutil.make_archive(
            M3_EXPORT_DIR.rstrip("/\\"),
            "zip",
            root_dir=M3_EXPORT_DIR
        )
    else:
        m3_zip_path = "shutil belum tersedia; file CSV/JSON sudah disimpan."

    print("\nFile hasil M3 disimpan:")
    print(f"- {m3_report_path}")
    print(f"- {m3_detail_path}")
    print(f"- {m3_json_path}")
    print(f"- {m3_zip_path}")


Tabel laporan M3 masih kosong karena belum ada kasus yang siap diuji.


,Query,P(kandidat benar),P(pengecoh),Lolos?
